# California Grid Stress Monitoring Dashboard

**Analytical Notebook: EIA-930 California Balancing Authority Grid Operations | January–April 2026**

| Field | Details |
| --- | --- |
| Author | Sileshi Hirpa |
| Date | April-May 2026 |
| Data source | EIA Form EIA-930 - Balancing Authority Hourly Operations |
| Scope | California balancing authorities only: BANC, CISO, IID, LDWP, TIDC |
| Tools | Python, pandas, Plotly, Jupyter Notebook, Tableau Public |

---

## Executive Summary

This notebook presents a portfolio-ready analysis of public California electricity
grid operations data from EIA Form EIA-930. The analysis prepares hourly balancing
authority records, validates the California-only scope, converts timestamps to
Pacific Time, engineers a Stress Index, classifies review-priority hours, and
exports dashboard-ready datasets for Tableau Public.

The central analytical goal is to identify periods when electricity demand
approaches the observed peak level for each California balancing authority. Because
California balancing authorities operate at very different scales, the Stress Index
normalizes each authority against its own observed peak demand within the
January-April 2026 dataset window.

The notebook supports the public portfolio README by making the full analytical
workflow reproducible: raw data preparation, scope validation, feature engineering,
review-priority classification, key findings, limitations, and dashboard data model
export.

---

## Table of Contents

1. Business Question and Project Goals
2. Setup: Imports, Paths, and Constants
3. Data Loading and Initial Inspection
4. Data Cleaning and Timezone Conversion
5. Scope Validation and California Filtering
6. California Demand Visualization
7. Stress Index Feature Engineering
8. Review Priority Classification
9. Key Findings
10. Top Review Hours
11. Operational Context
12. Processed Output Export
13. Dashboard Data Model Export
14. Limitations and Caveats
15. Conclusion and Portfolio Summary

---

## Methodology

The notebook follows a reproducible analytics workflow:

1. **Data ingestion:** Load the public EIA-930 hourly balancing authority file
   using relative file paths.
2. **Data cleaning:** Standardize key column names, remove incomplete records,
   remove duplicate authority-hour records, convert UTC timestamps to Pacific Time,
   and coerce measurement fields to numeric values.
3. **Scope validation:** Inspect the balancing authorities present in the raw file,
   identify that the source contains records beyond California, and filter the
   working dataset to BANC, CISO, IID, LDWP, and TIDC.
4. **Visualization:** Build time-series views that compare California balancing
   authority demand patterns and reveal differences in scale and daily demand
   behavior.
5. **Feature engineering:** Compute `peak_demand_mw` for each balancing authority
   and calculate `stress_index = (demand_mw / peak_demand_mw) × 100`.
6. **Priority classification:** Use named threshold constants to classify each hour
   into High, Medium, or Low Review Priority.
7. **Evidence summary:** Compute key findings directly from the data, including
   priority-hour counts, peak demand, and average Stress Index.
8. **Dashboard export:** Create dashboard-ready detail and summary files for
   Tableau Public.
9. **Limitations:** Document the dataset window, public-data scope,
   non-historical peak denominator, and the fact that this is not an enterprise
   reliability model.

---

## Project Overview

This project uses publicly available EIA-930 data from the U.S. Energy Information
Administration and does not represent any utility's internal operational systems,
enterprise risk models, internal safety frameworks, or proprietary data pipelines.
The Stress Index denominator is derived from the January-April 2026 dataset window
only - not from a historical all-time record.

## Business Question

How can hourly electricity demand data be prepared, validated, and visualized so
that a reviewer can quickly identify periods that may deserve additional
operational attention?

## Project Goals

This notebook demonstrates:

1. Loading and inspecting publicly available hourly electricity demand data.
2. Cleaning and standardizing key fields for consistent downstream use.
3. Validating that the dataset scope matches the California-focused analysis objective.
4. Building time-series visualizations of demand patterns across balancing authorities.
5. Engineering a Stress Index based on demand relative to observed peak demand.
6. Classifying each hour into a review priority tier using the Stress Index.
7. Summarizing the highest-priority hours in a compact, decision-ready table.
8. Exporting processed and dashboard-ready datasets for use in Tableau and Power BI.

## Why This Analysis Matters

Grid reliability depends on understanding when demand approaches observed peak
levels. A reviewer who can quickly identify high-stress hours - and understand
whether local generation or imported energy was meeting that demand - is better
positioned to investigate potential reliability concerns. This notebook provides
that structure through systematic data preparation, feature engineering, and a
priority-ranked output table.

In [1]:
# ============================================
# STEP 1: IMPORT LIBRARIES
# ============================================

import pandas as pd
import numpy as np
import plotly.express as px
import os

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)

print("Libraries imported successfully.")

Libraries imported successfully.


## Step 1 Explanation: Importing Libraries

Before working with the dataset, I import the Python libraries I need for:

- reading the CSV file
- cleaning and transforming the data
- computing simple summary metrics
- creating interactive charts

This is a standard first step in a Jupyter notebook workflow.


In [2]:
# ============================================
# STEP 2: DEFINE FILE PATHS AND CONSTANTS
# ============================================

DATA_PATH     = "../data/raw/EIA930_BALANCE_2026_Jan_Jun.csv"
PROCESSED_DIR = "../data/processed"
OUTPUT_DIR    = "../outputs"

os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Review priority classification thresholds
# Named constants make threshold values easy to locate and adjust
HIGH_REVIEW_THRESHOLD   = 90   # Stress Index >= 90  ->  High Review Priority
MEDIUM_REVIEW_THRESHOLD = 75   # Stress Index >= 75  ->  Medium Review Priority
TOP_N_REVIEW_HOURS      = 10   # Rows shown in the ranked review table

print("File paths configured.")
print("Output directories ready.")
print(f"Review thresholds: High >= {HIGH_REVIEW_THRESHOLD}, Medium >= {MEDIUM_REVIEW_THRESHOLD}")
print(f"Top review hours to display: {TOP_N_REVIEW_HOURS}")

File paths configured.
Output directories ready.
Review thresholds: High >= 90, Medium >= 75
Top review hours to display: 10


## Step 2: File Paths and Configuration

This cell defines:

- the source data file path
- the output directories for processed data and summary tables
- named constants for review priority thresholds and display settings

Using named constants (`HIGH_REVIEW_THRESHOLD`, `MEDIUM_REVIEW_THRESHOLD`,
`TOP_N_REVIEW_HOURS`) makes the classification logic readable and the threshold
values easy to find and adjust without editing the function body directly. They
are defined here - in the configuration section - so any reader encounters them
before the classification step.

The `os.makedirs(exist_ok=True)` calls ensure output directories exist at
runtime, preventing export errors if they have not been created manually.

### Step 3: Load and Inspect Raw Dataset

The first technical step in the analysis is to load the raw EIA-930 balancing authority dataset into a pandas DataFrame. At this stage, the dataset is intentionally kept in its original structure. No filtering, aggregation, renaming, or transformation is performed before the initial inspection.

This step serves four purposes. First, it confirms that the file path is correct and that the dataset can be read successfully. Second, it verifies the number of rows and columns in the source file. Third, it displays the original column names so that the structure of the raw data is documented before any cleaning takes place. Finally, it helps identify which fields are most relevant for the California grid analysis, including balancing authority, date and time fields, electricity demand, demand forecast, net generation, and total interchange.

Inspecting the raw dataset before applying transformations is an important data quality practice. It helps detect structural issues early, such as unexpected column names, inconsistent formatting, missing fields, or columns that may require standardization before analysis. This creates a clear audit trail from the original source file to the cleaned analytical dataset used later in the notebook.

### Step 3B: Standardize Known Raw Column-Name Inconsistencies

After inspecting the raw column names, two minor naming inconsistencies were identified in the source dataset. One column contained an extra space before the adjusted label, and another column appeared to contain a spelling inconsistency in the phrase describing solar generation with integrated battery storage.

These issues do not affect the underlying energy values, but they can create problems later in the notebook if column names are referenced directly in code. For example, small differences in spacing or spelling can cause selection, renaming, aggregation, or dashboard export steps to fail even when the intended data column is present.

To prevent these issues, the affected column names are standardized before the main analysis columns are renamed. This keeps the raw data values unchanged while making the dataset easier to work with programmatically. The correction is limited to column labels only and does not modify any measured electricity demand, generation, or interchange values.

This step improves reproducibility, reduces the risk of downstream errors, and ensures that any dashboard-ready outputs use clean and consistent field names.

In [5]:
# ============================================
# STEP 3: LOAD THE DATA
# ============================================

df = pd.read_csv(DATA_PATH, low_memory=False)

print("Dataset loaded successfully.")
print("Number of rows and columns:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

Dataset loaded successfully.
Number of rows and columns: (158182, 65)

Column names:
['Balancing Authority', 'Data Date', 'Hour Number', 'Local Time at End of Hour', 'UTC Time at End of Hour', 'Demand Forecast (MW)', 'Demand (MW)', 'Net Generation (MW)', 'Total Interchange (MW)', 'Sum(Valid DIBAs) (MW)', 'Demand (MW) (Imputed)', 'Net Generation (MW) (Imputed)', 'Total Interchange (MW) (Imputed)', 'Demand (MW) (Adjusted)', 'Net Generation (MW) (Adjusted)', 'Total Interchange (MW) (Adjusted)', 'Net Generation (MW) from Coal', 'Net Generation (MW) from Natural Gas', 'Net Generation (MW) from Nuclear', 'Net Generation (MW) from All Petroleum Products', 'Net Generation (MW) from Hydropower Excluding Pumped Storage', 'Net Generation (MW) from Pumped Storage', 'Net Generation (MW) from Solar without Integrated Battery Storage', 'Net Generation (MW) from Solar with Integrated Battery Storage', 'Net Generation (MW) from Wind without Integrated Battery Storage', 'Net Generation (MW) from Wind wi

In [6]:
# Preview the first few rows
df.head()

,Balancing Authority,Data Date,Hour Number,Local Time at End of Hour,UTC Time at End of Hour,Demand Forecast (MW),Demand (MW),Net Generation (MW),Total Interchange (MW),Sum(Valid DIBAs) (MW),Demand (MW) (Imputed),Net Generation (MW) (Imputed),Total Interchange (MW) (Imputed),Demand (MW) (Adjusted),Net Generation (MW) (Adjusted),Total Interchange (MW) (Adjusted),Net Generation (MW) from Coal,Net Generation (MW) from Natural Gas,Net Generation (MW) from Nuclear,Net Generation (MW) from All Petroleum Products,Net Generation (MW) from Hydropower Excluding Pumped Storage,Net Generation (MW) from Pumped Storage,Net Generation (MW) from Solar without Integrated Battery Storage,Net Generation (MW) from Solar with Integrated Battery Storage,Net Generation (MW) from Wind without Integrated Battery Storage,Net Generation (MW) from Wind with Integrated Battery Storage,Net Generation (MW) from Battery Storage,Net Generation (MW) from Other Energy Storage,Net Generation (MW) from Unknown Energy Storage,Net Generation (MW) from Geothermal,Net Generation (MW) from Other Fuel Sources,Net Generation (MW) from Unknown Fuel Sources,Net Generation (MW) from Coal (Imputed),Net Generation (MW) from Natural Gas (Imputed),Net Generation (MW) from Nuclear (Imputed),Net Generation (MW) from All Petroleum Products (Imputed),Net Generation (MW) from Hydropower Excluding Pumped Storage (Imputed),Net Generation (MW) from Pumped Storage (Imputed),Net Generation (MW) from Solar without Integrated Battery Storage (Imputed),Net Generation (MW) from Solar with Integrated Battery Storage (Imputed),Net Generation (MW) from Wind without Integrated Battery Storage (Imputed),Net Generation (MW) from Wind with Integrated Battery Storage (Imputed),Net Generation (MW) from Battery Storage (Imputed),Net Generation (MW) from Other Energy Storage (Imputed),Net Generation (MW) from Unknown Energy Storage (Imputed),Net Generation (MW) from Geothermal (Imputed),Net Generation (MW) from Other Fuel Sources (Imputed),Net Generation (MW) from Unknown Fuel Sources (Imputed),Net Generation (MW) from Coal (Adjusted),Net Generation (MW) from Natural Gas (Adjusted),Net Generation (MW) from Nuclear (Adjusted),Net Generation (MW) from All Petroleum Products (Adjusted),Net Generation (MW) from Hydropower Excluding Pumped Storage (Adjusted),Net Generation (MW) from Pumped Storage (Adjusted),Net Generation (MW) from Solar without Integrated Battery Storage (Adjusted),Net Generation (MW) from Solar witho Integrated Battery Storage (Adjusted),Net Generation (MW) from Wind without Integrated Battery Storage (Adjusted),Net Generation (MW) from Wind with Integrated Battery Storage (Adjusted),Net Generation (MW) from Battery Storage (Adjusted),Net Generation (MW) from Other Energy Storage (Adjusted),Net Generation (MW) from Unknown Energy Storage (Adjusted),Net Generation (MW) from Geothermal (Adjusted),Net Generation (MW) from Other Fuel Sources (Adjusted),Net Generation (MW) from Unknown Fuel Sources (Adjusted),Region
0,AECI,01/01/2026,1,01/01/2026 1:00:00 AM,01/01/2026 7:00:00 AM,2706.00,2625.00,2942.00,317.00,317.00,NaN,NaN,NaN,2625.00,2942.00,317.00,1711.00,629.00,NaN,NaN,NaN,NaN,NaN,NaN,602.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1711.00,629.00,NaN,NaN,NaN,NaN,NaN,NaN,602.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MIDW
1,AECI,01/01/2026,2,01/01/2026 2:00:00 AM,01/01/2026 8:00:00 AM,2638.00,2618.00,2895.00,277.00,277.00,NaN,NaN,NaN,2618.00,2895.00,277.00,1786.00,699.00,NaN,NaN,NaN,NaN,NaN,NaN,410.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1786.00,699.00,NaN,NaN,NaN,NaN,NaN,NaN,410.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MIDW
2,AECI,01/01/2026,3,01/01/2026 3:00:00 AM,01/01/2026 9:00:00 AM,2627.00,2600.00,3036.00,436.00,436.00,NaN,NaN,NaN,2600.00,3036.00,436.00,2137.00,637.00,NaN,NaN,NaN,NaN,NaN,NaN,262.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2137.00,637

In [7]:
# Check for possible column-name issues
for col in df.columns:
    if "  " in col or "witho" in col:
        print(col)

Net Generation (MW) from Solar without Integrated Battery Storage
Net Generation (MW) from Wind without Integrated Battery Storage
Net Generation (MW) from Solar without Integrated Battery Storage (Imputed)
Net Generation (MW) from Wind without Integrated Battery Storage (Imputed)
Net Generation (MW) from Pumped Storage  (Adjusted)
Net Generation (MW) from Solar without Integrated Battery Storage (Adjusted)
Net Generation (MW) from Solar witho Integrated Battery Storage (Adjusted)
Net Generation (MW) from Wind without Integrated Battery Storage (Adjusted)


In [8]:
# Standardize known raw EIA-930 column-name inconsistencies
df = df.rename(columns={
    "Net Generation (MW) from Pumped Storage  (Adjusted)":
        "Net Generation (MW) from Pumped Storage (Adjusted)",

    "Net Generation (MW) from Solar witho Integrated Battery Storage (Adjusted)":
        "Net Generation (MW) from Solar with Integrated Battery Storage (Adjusted)"
})

In [9]:
# Confirm the corrected columns exist
columns_to_check = [
    "Net Generation (MW) from Pumped Storage (Adjusted)",
    "Net Generation (MW) from Solar with Integrated Battery Storage (Adjusted)"
]

for col in columns_to_check:
    print(col, ":", col in df.columns)

Net Generation (MW) from Pumped Storage (Adjusted) : True
Net Generation (MW) from Solar with Integrated Battery Storage (Adjusted) : True


**Column-name standardization only:**  
This cell corrects minor spacing and spelling inconsistencies in selected raw column labels. It does not alter the numerical data values.

### Step 4: Rename Important Columns for Analysis

The raw EIA-930 dataset contains long column names with spaces, parentheses, and measurement labels. These names are useful for understanding the source file, but they are not ideal for repeated analysis in Python. To make the notebook easier to read and less error-prone, the main fields used in this project are renamed using shorter, consistent snake_case names.

This step focuses only on the core columns needed for the California grid analysis, including balancing authority, date and time fields, electricity demand, demand forecast, net generation, and total interchange. The remaining source columns are left unchanged unless they are needed later.

Renaming columns at this stage improves code readability and creates a cleaner foundation for later filtering, aggregation, visualization, and dashboard export. The operation changes column labels only. It does not change the underlying data values.

In [10]:
# ============================================
# STEP 4: RENAME IMPORTANT COLUMNS
# ============================================

rename_dict = {
    "Balancing Authority": "balancing_authority",
    "Data Date": "data_date",
    "Hour Number": "hour_number",
    "UTC Time at End of Hour": "utc_time_at_end_of_hour",
    "Demand Forecast (MW)": "demand_forecast_mw",
    "Demand (MW)": "demand_mw",
    "Net Generation (MW)": "net_generation_mw",
    "Total Interchange (MW)": "total_interchange_mw"
}

df.rename(columns=rename_dict, inplace=True)

print("Columns renamed successfully.")
print("Dataset shape after renaming:", df.shape)

print("\nRenamed analysis columns:")
for old_col, new_col in rename_dict.items():
    print(f"{old_col}  ->  {new_col}")

print("\nFirst 10 columns after renaming:")
print(df.columns.tolist()[:10])

Columns renamed successfully.
Dataset shape after renaming: (158182, 65)

Renamed analysis columns:
Balancing Authority  ->  balancing_authority
Data Date  ->  data_date
Hour Number  ->  hour_number
UTC Time at End of Hour  ->  utc_time_at_end_of_hour
Demand Forecast (MW)  ->  demand_forecast_mw
Demand (MW)  ->  demand_mw
Net Generation (MW)  ->  net_generation_mw
Total Interchange (MW)  ->  total_interchange_mw

First 10 columns after renaming:
['balancing_authority', 'data_date', 'hour_number', 'Local Time at End of Hour', 'utc_time_at_end_of_hour', 'demand_forecast_mw', 'demand_mw', 'net_generation_mw', 'total_interchange_mw', 'Sum(Valid DIBAs) (MW)']


In [11]:
required_columns = [
    "balancing_authority",
    "data_date",
    "hour_number",
    "utc_time_at_end_of_hour",
    "demand_forecast_mw",
    "demand_mw",
    "net_generation_mw",
    "total_interchange_mw"
]

missing_columns = [col for col in required_columns if col not in df.columns]

if len(missing_columns) == 0:
    print("All required analysis columns are present.")
else:
    print("Missing required columns:", missing_columns)

All required analysis columns are present.


### Step 4 Summary

The key source columns were successfully renamed into shorter, consistent analysis-friendly names. The dataset shape remained unchanged at **158,182 rows and 65 columns**, confirming that the operation changed column labels only and did not remove or alter any records.

The required fields for the core California grid analysis are now available, including balancing authority, date, hour, UTC timestamp, demand forecast, actual demand, net generation, and total interchange. These columns provide the foundation for later time-based analysis, demand and generation comparisons, regional filtering, and dashboard-ready exports.

The validation check confirmed that all required analysis columns are present. This means the notebook can safely proceed to the next stage of data type conversion, cleaning, and preparation.

### Step 5: Basic Data Cleaning

After the key columns have been renamed, the dataset is prepared for analysis by applying a small set of basic cleaning steps. This includes removing records that are missing essential fields, dropping duplicate hourly records for the same balancing authority, converting date and time fields into proper datetime formats, and converting key electricity measures into numeric values.

The cleaning is intentionally limited at this stage. The goal is not to reshape or aggregate the dataset yet, but to make sure the core fields are reliable enough for time-based analysis, filtering, visualization, and dashboard preparation.

A Pacific time column is also created from the UTC timestamp because the project focuses on California grid conditions. This makes later analysis easier to interpret in the regional time zone most relevant to the dashboard.

In [12]:
# ============================================
# STEP 5: BASIC DATA CLEANING
# ============================================

# Remove records missing essential fields for analysis
df = df.dropna(
    subset=[
        "balancing_authority",
        "utc_time_at_end_of_hour",
        "demand_mw"
    ]
)

# Remove duplicate hourly records for the same balancing authority
df = df.drop_duplicates(
    subset=[
        "balancing_authority",
        "utc_time_at_end_of_hour"
    ]
)

# Convert date and timestamp fields
df["data_date"] = pd.to_datetime(df["data_date"], errors="coerce")

df["utc_time_at_end_of_hour"] = pd.to_datetime(
    df["utc_time_at_end_of_hour"],
    utc=True,
    errors="coerce"
)

# Convert UTC time to Pacific time for California-focused analysis
df["local_time_pacific"] = df["utc_time_at_end_of_hour"].dt.tz_convert(
    "America/Los_Angeles"
)

# Convert core electricity measures to numeric values
numeric_cols = [
    "demand_mw",
    "demand_forecast_mw",
    "net_generation_mw",
    "total_interchange_mw"
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

print("Basic cleaning complete.")
print("New dataset shape:", df.shape)

print("\nData types for core analysis columns:")
print(df[
    [
        "balancing_authority",
        "data_date",
        "utc_time_at_end_of_hour",
        "local_time_pacific",
        "demand_mw",
        "demand_forecast_mw",
        "net_generation_mw",
        "total_interchange_mw"
    ]
].dtypes)

Basic cleaning complete.
New dataset shape: (137194, 66)

Data types for core analysis columns:
balancing_authority                                     object
data_date                                       datetime64[ns]
utc_time_at_end_of_hour                    datetime64[ns, UTC]
local_time_pacific         datetime64[ns, America/Los_Angeles]
demand_mw                                              float64
demand_forecast_mw                                     float64
net_generation_mw                                      float64
total_interchange_mw                                   float64
dtype: object


### Step 5 Summary

The basic cleaning step reduced the dataset from **158,182 rows and 65 columns** to **137,194 rows and 66 columns**. The row count decreased because records with missing essential fields and duplicate balancing-authority hourly records were removed. The column count increased by one because a new `local_time_pacific` field was created for California-focused time analysis.

The core date and time fields were successfully converted into datetime formats. The UTC timestamp is now stored as a timezone-aware datetime field, and the Pacific time version is available for regional interpretation. This is important because California grid demand patterns are strongly tied to local daily cycles, including morning, afternoon, and evening demand periods.

The main electricity measures were also converted to numeric format:

```text
demand_mw
demand_forecast_mw
net_generation_mw
total_interchange_mw

### Step 6: Review the Full Dataset

After basic cleaning, the full dataset is reviewed to confirm its structure, data types, and numerical ranges. This step provides a broad quality check before the analysis moves into filtering, feature creation, and visualization.

The `info()` output shows the number of records, column count, non-null values, and data types for each field. This helps confirm that the date, time, and numeric columns were converted correctly in the previous step.

The descriptive statistics table summarizes the main electricity measures used in the analysis: actual demand, demand forecast, net generation, and total interchange. These summary values help identify the scale of the data and reveal possible issues such as extreme values, missing values, or unusual ranges that may need closer review before dashboard preparation.

In [13]:
# ============================================
# STEP 6: REVIEW THE FULL DATASET
# ============================================

print("Summary of the dataset:")
df.info()

print("\nBasic statistics for core electricity measures:")

core_numeric_cols = [
    "demand_mw",
    "demand_forecast_mw",
    "net_generation_mw",
    "total_interchange_mw"
]

df[core_numeric_cols].describe()

Summary of the dataset:
<class 'pandas.core.frame.DataFrame'>
Index: 137194 entries, 0 to 155566
Data columns (total 66 columns):
 #   Column                                                                        Non-Null Count   Dtype                              
---  ------                                                                        --------------   -----                              
 0   balancing_authority                                                           137194 non-null  object                             
 1   data_date                                                                     137194 non-null  datetime64[ns]                     
 2   hour_number                                                                   137194 non-null  int64                              
 3   Local Time at End of Hour                                                     137194 non-null  object                             
 4   utc_time_at_end_of_hour                          

,demand_mw,demand_forecast_mw,net_generation_mw,total_interchange_mw
count,137194.00,136641.00,137194.00,137047.00
mean,9135.88,8999.31,9005.80,-134.70
std,20026.00,18790.75,20330.58,1517.27
min,-40972.00,-25409.00,-24989.00,-21604.00
25%,768.00,740.00,803.00,-583.00
50%,2423.50,2160.00,1883.00,-63.00
75%,6342.00,5910.00,6995.00,390.00
max,2576875.00,271191.00,2576906.00,43330.00


### Step 6 Summary

The full dataset review confirmed that the cleaned dataset contains **137,194 rows and 66 columns**. The main date and time fields were converted successfully, including a UTC timestamp and a Pacific time version for California-focused analysis. The core electricity measures were also converted to numeric format, which allows the project to move into filtering, aggregation, and visualization.

The review also identified several data quality items that require attention before dashboard preparation. The `demand_forecast_mw` and `total_interchange_mw` fields contain some missing values, although the missing counts are small relative to the full dataset. More importantly, the descriptive statistics show unusually large maximum values for `demand_mw` and `net_generation_mw`. These values are much larger than expected for ordinary hourly balancing authority records and should be inspected before final visualizations are created.

This step confirms that the dataset is structurally ready for analysis, but it also shows why an outlier review is necessary before producing portfolio-ready charts and dashboard exports.

In [14]:
# ============================================
# STEP 6B: INSPECT EXTREME VALUES
# ============================================

extreme_cols = [
    "demand_mw",
    "demand_forecast_mw",
    "net_generation_mw",
    "total_interchange_mw"
]

for col in extreme_cols:
    print(f"\nTop 10 largest values for {col}:")
    display(
        df[
            [
                "balancing_authority",
                "data_date",
                "hour_number",
                "utc_time_at_end_of_hour",
                "local_time_pacific",
                col
            ]
        ]
        .sort_values(col, ascending=False)
        .head(10)
    )

    print(f"\nTop 10 smallest values for {col}:")
    display(
        df[
            [
                "balancing_authority",
                "data_date",
                "hour_number",
                "utc_time_at_end_of_hour",
                "local_time_pacific",
                col
            ]
        ]
        .sort_values(col, ascending=True)
        .head(10)
    )


Top 10 largest values for demand_mw:



Top 10 smallest values for demand_mw:



Top 10 largest values for demand_forecast_mw:



Top 10 smallest values for demand_forecast_mw:



Top 10 largest values for net_generation_mw:



Top 10 smallest values for net_generation_mw:



Top 10 largest values for total_interchange_mw:



Top 10 smallest values for total_interchange_mw:


### Step 6B Summary: Outlier Review

The outlier review identified several extreme records in the core electricity measures. The largest demand and net generation values came from the WACM balancing authority, including records above one million megawatts. These values are far outside the expected range for ordinary hourly balancing authority operations and are likely raw-data anomalies or reporting issues.

The review also showed that some large values, such as PJM demand and generation above 130,000 megawatts, may be reasonable because PJM is a very large balancing authority. Therefore, the outlier strategy should not rely on a single universal cutoff for every region.

For the California-focused portion of the project, the CISO interchange values are especially important. Several CISO records show large negative total interchange values, which may reflect substantial imports into the California grid rather than data errors. These records should be retained unless a specific quality rule shows that they are invalid.

Based on this review, the project should avoid allowing extreme non-California anomalies to distort dashboard visuals. The preferred approach is to focus the dashboard analysis on California-relevant balancing authorities and apply any outlier handling transparently.

In [15]:
# ============================================
# STEP 6C: REVIEW CALIFORNIA-FOCUSED RECORDS
# ============================================

california_bas = ["CISO"]

df_ca_check = df[df["balancing_authority"].isin(california_bas)].copy()

print("California-focused dataset shape:", df_ca_check.shape)

print("\nSummary statistics for CISO:")
display(
    df_ca_check[
        [
            "demand_mw",
            "demand_forecast_mw",
            "net_generation_mw",
            "total_interchange_mw"
        ]
    ].describe()
)

print("\nTop 10 largest CISO demand values:")
display(
    df_ca_check[
        [
            "balancing_authority",
            "data_date",
            "hour_number",
            "utc_time_at_end_of_hour",
            "local_time_pacific",
            "demand_mw",
            "demand_forecast_mw",
            "net_generation_mw",
            "total_interchange_mw"
        ]
    ]
    .sort_values("demand_mw", ascending=False)
    .head(10)
)

print("\nTop 10 smallest CISO total interchange values:")
display(
    df_ca_check[
        [
            "balancing_authority",
            "data_date",
            "hour_number",
            "utc_time_at_end_of_hour",
            "local_time_pacific",
            "demand_mw",
            "demand_forecast_mw",
            "net_generation_mw",
            "total_interchange_mw"
        ]
    ]
    .sort_values("total_interchange_mw", ascending=True)
    .head(10)
)

California-focused dataset shape: (2585, 66)

Summary statistics for CISO:



Top 10 largest CISO demand values:



Top 10 smallest CISO total interchange values:


### Step 6C Summary: California-Focused Data Review

The California-focused subset contains **2,585 CISO hourly records** after basic cleaning. Unlike the full national dataset, the CISO records do not show extreme million-megawatt anomalies. The demand values range from **18,748 MW** to **35,596 MW**, with an average demand of about **24,332 MW**. This range is reasonable for a California ISO hourly demand analysis.

The CISO demand peaks occur in the early evening Pacific time, especially between **5:00 PM and 8:00 PM**. This timing is important because it aligns with the period when electricity demand often rises as households and businesses continue using power while solar generation begins to decline.

The total interchange values are mostly negative, with an average of about **-5,635 MW**. In this dataset, negative interchange values suggest that CISO is often relying on net imports during the observed period. The most negative interchange values occur during early morning hours in January, reaching approximately **-11,236 MW**.

This review confirms that the CISO subset is suitable for the California grid dashboard. Since the extreme outliers found in the full dataset are mainly from non-California balancing authorities, the dashboard should focus on CISO records or clearly separate California analysis from broader regional comparisons.

### Step 7: Create the California Analysis Dataset

The raw EIA-930 file contains records for many balancing authorities, not only California. Since this project focuses on California grid conditions, the analysis is narrowed to the California ISO balancing authority, identified in the dataset as `CISO`.

This step creates a dedicated California analysis DataFrame named `df_ca`. Separating the California subset from the full national dataset helps prevent non-California records and national-scale outliers from distorting the project’s visuals, findings, and dashboard exports.

The full dataset is still useful for context and quality checks, but the main analytical work will use the CISO subset. This keeps the project aligned with its stated purpose: analyzing California electricity demand, generation, and interchange patterns.

In [16]:
# ============================================
# STEP 7: CREATE THE CALIFORNIA ANALYSIS DATASET
# ============================================

df_ca = df[df["balancing_authority"] == "CISO"].copy()

print("California analysis dataset created.")
print("Dataset shape:", df_ca.shape)

print("\nDate range:")
print("Start:", df_ca["local_time_pacific"].min())
print("End:", df_ca["local_time_pacific"].max())

print("\nCore measure summary:")
display(
    df_ca[
        [
            "demand_mw",
            "demand_forecast_mw",
            "net_generation_mw",
            "total_interchange_mw"
        ]
    ].describe()
)

California analysis dataset created.
Dataset shape: (2585, 66)

Date range:
Start: 2026-01-01 01:00:00-08:00
End: 2026-04-20 00:00:00-07:00

Core measure summary:


### Step 7 Summary

The California analysis dataset was created successfully by filtering the full EIA-930 dataset to the `CISO` balancing authority. The resulting dataset contains **2,585 rows and 66 columns**, covering hourly records from **January 1, 2026** through **April 20, 2026** in Pacific time.

The summary statistics show that CISO demand ranges from **18,748 MW** to **35,596 MW**, with an average of about **24,332 MW**. These values are reasonable for California ISO hourly demand and do not show the extreme million-megawatt anomalies found in the broader national dataset.

The total interchange field has an average of about **-5,635 MW**, suggesting that CISO frequently appears as a net importer during the observed period. Because this subset is clean and aligned with the project purpose, it should be used as the main dataset for California-focused visualizations, feature creation, and dashboard exports.

## Exploratory Data Analysis: California Grid Demand and Supply Patterns

The next section validates the project scope and develops the main exploratory visuals needed before feature engineering. Since this project is a statewide California grid analysis, the working dataset should include the California balancing authorities available in the EIA-930 file, not only California ISO.

The full EIA-930 dataset is first reviewed for scope and outliers. Then the data is filtered to the California balancing authorities used in this project: BANC, CISO, IID, LDWP, and TIDC. After that, several visualizations examine hourly demand, daily demand trends, forecast accuracy, generation balance, interchange behavior, time-of-day demand patterns, and day-of-week variation.

These steps create the analytical foundation for the next phase: engineering a grid stress index.

### Step 8: Scope Validation and Outlier Review

The first full-dataset visualization showed an unusual pattern: most values appeared compressed near the bottom of the chart while a few extreme values stretched the y-axis upward. This suggested that the issue was not the chart itself, but the analytical scope of the data.

The EIA-930 file contains balancing authorities from across the United States. Since this project focuses on California grid conditions, the full dataset must be reviewed before interpretation. This step checks which balancing authorities are present and inspects the highest and lowest demand records before creating a California-only working dataset.

A chart can run without error and still be misleading if the data does not match the intended business question. Scope validation prevents the notebook from drawing conclusions from the wrong analytical population.

In [17]:
# ============================================
# STEP 8: SCOPE VALIDATION AND OUTLIER REVIEW
# ============================================

review_cols = [
    "balancing_authority",
    "local_time_pacific",
    "demand_mw",
    "demand_forecast_mw",
    "net_generation_mw",
    "total_interchange_mw"
]

print("Balancing authorities in the full dataset:")
print(sorted(df["balancing_authority"].dropna().unique()))

print("\nTop 20 highest demand records:")
display(
    df.sort_values("demand_mw", ascending=False)[review_cols].head(20)
)

print("\nTop 20 lowest demand records:")
display(
    df.sort_values("demand_mw", ascending=True)[review_cols].head(20)
)

Balancing authorities in the full dataset:
['AECI', 'AVA', 'AZPS', 'BANC', 'BHBA', 'BPAT', 'CHPD', 'CISO', 'CPLE', 'CPLW', 'DOPD', 'DUK', 'EPE', 'ERCO', 'FMPP', 'FPC', 'FPL', 'GCPD', 'GVL', 'HST', 'IID', 'IPCO', 'ISNE', 'JEA', 'LDWP', 'LGEE', 'MISO', 'NEVP', 'NWMT', 'NYIS', 'PACE', 'PACW', 'PGE', 'PJM', 'PNM', 'PSCO', 'PSEI', 'SC', 'SCEG', 'SCL', 'SEC', 'SOCO', 'SPA', 'SRP', 'SWPP', 'TAL', 'TEC', 'TEPC', 'TIDC', 'TPWR', 'TVA', 'WACM', 'WALC', 'WAUW']

Top 20 highest demand records:



Top 20 lowest demand records:


### Step 8 Summary

The scope validation step confirmed that the full EIA-930 dataset contains balancing authorities from across the United States, not only California. The full dataset includes authorities such as PJM, NYIS, MISO, TVA, SWPP, CISO, BANC, IID, LDWP, and TIDC.

The outlier review also showed that the largest demand records are not from California. The two highest demand values come from WACM, with values above **1,000,000 MW** and **2,500,000 MW**, which are far outside the normal scale of hourly balancing authority demand. The next highest records are mostly from PJM, a large non-California balancing authority.

The lowest demand records also come from non-California authorities such as SEC, WALC, WAUW, FMPP, and PSCO. Several records contain negative demand values, which may reflect reporting issues, special balancing-authority accounting, or records outside the intended California analysis scope.

This confirms that the full dataset should not be used directly for the main California grid visualizations. The next step filters the data to the California balancing authorities: BANC, CISO, IID, LDWP, and TIDC.

In [18]:
# ============================================
# STEP 9: FILTER TO CALIFORNIA BALANCING AUTHORITIES
# ============================================

# California balancing authorities used in this statewide analysis
ca_bas = ["BANC", "CISO", "IID", "LDWP", "TIDC"]

# Create the main California-only working dataframe
df_ca = df[df["balancing_authority"].isin(ca_bas)].copy()

print("Original full dataset rows:", len(df))
print("California-only rows:", len(df_ca))

print("\nCalifornia balancing authorities in filtered data:")
print(sorted(df_ca["balancing_authority"].dropna().unique()))

print("\nRows by California balancing authority:")
display(
    df_ca["balancing_authority"]
    .value_counts()
    .rename_axis("balancing_authority")
    .reset_index(name="row_count")
)

Original full dataset rows: 137194
California-only rows: 13020

California balancing authorities in filtered data:
['BANC', 'CISO', 'IID', 'LDWP', 'TIDC']

Rows by California balancing authority:


### Step 9 Summary

The dataset was successfully filtered from the full EIA-930 file to the five California balancing authorities used in this statewide analysis: BANC, CISO, IID, LDWP, and TIDC.

The filtered California dataset contains **13,020 rows**, compared with **137,194 rows** in the cleaned full dataset. This confirms that the project scope has been narrowed from a national balancing-authority dataset to a California-focused working dataset.

The row counts are similar across the five California authorities, with each authority contributing roughly 2,585 to 2,615 records. This suggests that the dataset has consistent hourly coverage across the California authorities, although CISO and LDWP have slightly fewer records than IID and TIDC. The filtered dataset, `df_ca`, will be used for the remaining exploratory analysis, feature engineering, stress index construction, and dashboard export.

In [19]:
# ============================================
# STEP 10: RECHECK CALIFORNIA-ONLY DEMAND STATISTICS
# ============================================

core_cols = [
    "demand_mw",
    "demand_forecast_mw",
    "net_generation_mw",
    "total_interchange_mw"
]

print("California-only core measure summary:")
display(df_ca[core_cols].describe())

print("\nCore measure summary by balancing authority:")
display(
    df_ca.groupby("balancing_authority")[core_cols]
    .agg(["count", "mean", "min", "max"])
    .round(2)
)

California-only core measure summary:



Core measure summary by balancing authority:


### Step 10 Summary

The California-only dataset contains **13,020 hourly records** across the five selected balancing authorities. The overall demand values range from **0 MW** to **35,596 MW**, with an average of about **5,839 MW**. The maximum demand value now comes from the California-only scope and is no longer affected by the extreme non-California outliers found in the full dataset.

The by-authority summary shows clear scale differences across the California grid. CISO is the largest authority by demand, with an average demand of about **24,332 MW** and a maximum of **35,596 MW**. LDWP and BANC operate at smaller but still substantial demand levels, while IID and TIDC have much smaller demand profiles.

The interchange values also differ by authority. CISO has the largest average negative interchange, about **-5,635 MW**, which suggests frequent net imports during the observed period. BANC, LDWP, and TIDC also show negative average interchange, while IID shows positive average interchange. These differences show why statewide analysis should include all five authorities but also separate them visually when needed.

This step confirms that the California-only dataset is appropriate for the next stage of exploratory visualization. Because the authorities operate at very different scales, combined charts should be paired with panel views or authority-specific summaries to avoid misleading visual compression.

In [21]:
# ============================================
# STEP 11: COMBINED SCALE COMPARISON BY BALANCING AUTHORITY
# ============================================

fig_compare = px.line(
    df_ca.sort_values(["balancing_authority", "local_time_pacific"]),
    x="local_time_pacific",
    y="demand_mw",
    color="balancing_authority",
    title="California Electricity Demand by Balancing Authority: Scale Comparison",
    labels={
        "local_time_pacific": "Local Time (Pacific)",
        "demand_mw": "Demand (MW)",
        "balancing_authority": "Balancing Authority"
    }
)

fig_compare.show()

### Step 11 Summary

The combined California demand chart shows that CISO operates at a much larger demand scale than the other California balancing authorities. This makes the chart useful for comparing relative grid size across authorities.

However, the same scale difference also limits detailed interpretation. Smaller authorities such as IID and TIDC are visually compressed near the bottom of the chart, and even BANC and LDWP are difficult to read in detail. Therefore, this chart should be treated as a statewide scale comparison, not as the primary chart for authority-level demand patterns.

A panel view is needed next so that each authority can be reviewed separately with better readability.

In [28]:
# ============================================
# STEP 12: PANEL VIEW WITH CLEAN AVERAGE DEMAND LABELS
# ============================================

# Define the intended visual order from top to bottom
ba_order = ["BANC", "CISO", "IID", "LDWP", "TIDC"]

# Sort data for cleaner line plotting
df_ca_sorted = df_ca.sort_values(
    ["balancing_authority", "local_time_pacific"]
)

# Calculate average demand for each balancing authority
avg_demand_by_ba = (
    df_ca_sorted
    .groupby("balancing_authority", as_index=False)
    .agg(avg_demand_mw=("demand_mw", "mean"))
)

# Create panel chart
fig_panel = px.line(
    df_ca_sorted,
    x="local_time_pacific",
    y="demand_mw",
    facet_row="balancing_authority",
    category_orders={"balancing_authority": ba_order},
    title="Hourly Demand by California Balancing Authority with Average Reference Lines",
    labels={
        "local_time_pacific": "Local Time (Pacific)",
        "demand_mw": "Demand (MW)",
        "balancing_authority": "Balancing Authority"
    },
    height=1050
)

# Clean facet labels
fig_panel.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))

# Allow each panel to use its own y-axis scale
fig_panel.update_yaxes(matches=None)

# Plotly row numbering starts from the bottom.
n_panels = len(ba_order)

for ba in ba_order:
    avg_value = avg_demand_by_ba.loc[
        avg_demand_by_ba["balancing_authority"] == ba,
        "avg_demand_mw"
    ].iloc[0]

    # Convert top-to-bottom visual order into Plotly's bottom-to-top row numbering
    row_number = n_panels - ba_order.index(ba)

    # Add red dashed average line without overlapping text
    fig_panel.add_hline(
        y=avg_value,
        line_dash="dash",
        line_color="red",
        line_width=1.5,
        row=row_number,
        col=1
    )

    # Add clean label box near the top-right of the correct panel
    fig_panel.add_annotation(
        text=f"{ba} Avg Demand: {avg_value:,.0f} MW",
        xref="x domain",
        yref=f"y{row_number if row_number > 1 else ''} domain",
        x=0.985,
        y=0.90,
        showarrow=False,
        align="right",
        bgcolor="white",
        bordercolor="red",
        borderwidth=1,
        font=dict(size=11),
        row=row_number,
        col=1
    )

fig_panel.show()

### Step 12 Summary

The panel view improves the readability of California electricity demand patterns by separating each balancing authority into its own chart area. Each panel includes a red dashed reference line showing that authority’s average demand during the study period.

The reference lines make it easier to compare each authority’s hourly demand against its own normal operating level. This is important because the five California balancing authorities operate at very different scales. CISO has the largest average demand at about **24,332 MW**, while TIDC and IID operate at much smaller average demand levels.

The chart also reveals authority-specific behavior that was hidden in the combined chart. CISO shows the largest overall demand pattern, IID and TIDC show clear daily cycles at smaller scales, LDWP shows periods of very low reported demand at the beginning and end of the observed period, and BANC contains several sharp spikes that may deserve closer review before final dashboard interpretation.

This visual supports the next phase of analysis because it shows that raw demand alone is not enough for fair comparison across authorities. Later features, including the stress index, should be calculated within each balancing authority or normalized carefully before comparing statewide conditions.

### Step 13: Create Daily Demand Trend Views

The hourly panel chart provides detailed demand movement for each California balancing authority, but hourly data can be visually dense across several months. Daily summaries make the broader trend easier to interpret.

This step creates two daily views for each balancing authority: daily average demand and daily peak demand. Daily average demand smooths short-term hourly movement and shows the general demand level over time. Daily peak demand highlights the highest hourly load reached each day, which is more useful for identifying periods of operational pressure.

These daily views complement the hourly charts and prepare the notebook for the stress index, where high-demand periods need to be identified in a fair and consistent way across balancing authorities.

In [29]:
# ============================================
# STEP 13: CREATE DAILY DEMAND TREND VIEWS
# ============================================

# Create a daily summary table by balancing authority.
# Daily average demand shows the general daily load level.
# Daily peak demand shows the highest hourly demand reached each day.
daily_demand = (
    df_ca
    .assign(local_date=df_ca["local_time_pacific"].dt.date)
    .groupby(["local_date", "balancing_authority"], as_index=False)
    .agg(
        daily_avg_demand_mw=("demand_mw", "mean"),
        daily_peak_demand_mw=("demand_mw", "max")
    )
)

# Convert the local date field back to datetime for cleaner plotting.
daily_demand["local_date"] = pd.to_datetime(daily_demand["local_date"])

print("Daily demand summary created.")
print("Shape:", daily_demand.shape)

print("\nFirst five rows:")
display(daily_demand.head())

print("\nDaily demand summary by balancing authority:")
display(
    daily_demand
    .groupby("balancing_authority")[
        ["daily_avg_demand_mw", "daily_peak_demand_mw"]
    ]
    .agg(["mean", "min", "max"])
    .round(2)
)

Daily demand summary created.
Shape: (550, 4)

First five rows:



Daily demand summary by balancing authority:


### Step 13 Output Check

The daily demand summary was created successfully with **550 rows and 4 columns**. This is consistent with five California balancing authorities observed across the study period. Each row represents one balancing authority on one local date, with both average and peak demand calculated for that day.

The summary table confirms the large scale differences across California authorities. CISO has the highest daily average demand, with an average daily level of about **24,291 MW** and a maximum daily peak of **35,596 MW**. BANC and LDWP operate at smaller but still substantial levels, while IID and TIDC have much smaller daily demand profiles.

The daily peak results also reveal values that should remain visible in later interpretation. BANC has a maximum daily peak of **9,770 MW**, which is much higher than its typical daily average. LDWP shows very low minimum daily average and peak values, matching the earlier panel chart observation that LDWP has very low reported demand at the beginning and end of the period. These values do not stop the analysis, but they should be acknowledged when interpreting dashboard results.

The daily summary is ready for visualization through daily average demand and daily peak demand charts.

In [30]:
# ============================================
# STEP 13A: DAILY AVERAGE DEMAND CHART
# ============================================

fig_daily_avg = px.line(
    daily_demand,
    x="local_date",
    y="daily_avg_demand_mw",
    color="balancing_authority",
    title="Daily Average Electricity Demand by California Balancing Authority",
    labels={
        "local_date": "Date",
        "daily_avg_demand_mw": "Daily Average Demand (MW)",
        "balancing_authority": "Balancing Authority"
    }
)

fig_daily_avg.show()

In [31]:
# ============================================
# STEP 13B: DAILY PEAK DEMAND CHART
# ============================================

fig_daily_peak = px.line(
    daily_demand,
    x="local_date",
    y="daily_peak_demand_mw",
    color="balancing_authority",
    title="Daily Peak Electricity Demand by California Balancing Authority",
    labels={
        "local_date": "Date",
        "daily_peak_demand_mw": "Daily Peak Demand (MW)",
        "balancing_authority": "Balancing Authority"
    }
)

fig_daily_peak.show()

### Step 13 Summary

The daily demand trend views provide a smoother summary of California electricity demand than the hourly charts. The daily average chart shows general demand levels over time, while the daily peak chart highlights the highest hourly demand reached each day.

Both charts confirm that CISO dominates the statewide demand scale. CISO daily average demand generally stays far above the other California balancing authorities, and its daily peak demand reaches the highest levels in the dataset, especially during the mid-to-late March peak period.

The smaller authorities remain visible but compressed because they operate at much lower demand levels. BANC and LDWP show moderate daily demand compared with CISO, while IID and TIDC remain much smaller. The daily peak chart also shows several sharp BANC peaks, which may require closer review before final dashboard interpretation.

These charts are useful for statewide scale comparison. For authority-level interpretation, panel views or normalized metrics are more appropriate. This reinforces the need for the later stress index to be calculated within each balancing authority or normalized carefully before comparing grid stress across California.

#### Step 13C: Daily Peak Demand Panel View with Mean and Median Reference Lines

The combined daily peak chart is useful for statewide comparison, but the large scale differences across balancing authorities make smaller systems harder to interpret. A panel view improves readability by separating the authorities into individual charts.

To provide reference levels for interpretation, each panel includes two horizontal guide lines:
- a red dashed line for the mean daily peak demand
- a green dotted line for the median daily peak demand

Using both measures helps distinguish typical daily peak behavior from spike-driven behavior. If the mean is noticeably above the median, the authority may experience occasional unusually high peak days.

In [33]:
# ============================================
# STEP 13C: DAILY PEAK DEMAND PANEL VIEW
# WITH MEAN AND MEDIAN REFERENCE LINES
# ============================================

# Define visual order from top to bottom
ba_order = ["BANC", "CISO", "IID", "LDWP", "TIDC"]

# Calculate mean and median daily peak demand by balancing authority
daily_peak_stats = (
    daily_demand
    .groupby("balancing_authority", as_index=False)
    .agg(
        mean_daily_peak_mw=("daily_peak_demand_mw", "mean"),
        median_daily_peak_mw=("daily_peak_demand_mw", "median")
    )
)

# Create panel chart
fig_daily_peak_panel = px.line(
    daily_demand.sort_values(["balancing_authority", "local_date"]),
    x="local_date",
    y="daily_peak_demand_mw",
    facet_row="balancing_authority",
    category_orders={"balancing_authority": ba_order},
    title="Daily Peak Demand by California Balancing Authority: Panel View with Mean and Median Reference Lines",
    labels={
        "local_date": "Date",
        "daily_peak_demand_mw": "Daily Peak Demand (MW)",
        "balancing_authority": "Balancing Authority"
    },
    height=1050
)

# Clean facet labels
fig_daily_peak_panel.for_each_annotation(
    lambda a: a.update(text=a.text.split("=")[-1])
)

# Allow each panel to have its own y-axis scale
fig_daily_peak_panel.update_yaxes(matches=None)

# Plotly row numbering runs from bottom to top
n_panels = len(ba_order)

for ba in ba_order:
    row_number = n_panels - ba_order.index(ba)

    mean_value = daily_peak_stats.loc[
        daily_peak_stats["balancing_authority"] == ba,
        "mean_daily_peak_mw"
    ].iloc[0]

    median_value = daily_peak_stats.loc[
        daily_peak_stats["balancing_authority"] == ba,
        "median_daily_peak_mw"
    ].iloc[0]

    # Add mean reference line
    fig_daily_peak_panel.add_hline(
        y=mean_value,
        line_dash="dash",
        line_color="red",
        line_width=1.5,
        row=row_number,
        col=1
    )

    # Add median reference line
    fig_daily_peak_panel.add_hline(
        y=median_value,
        line_dash="dot",
        line_color="green",
        line_width=1.5,
        row=row_number,
        col=1
    )

    # Add annotation box in top-right corner
    fig_daily_peak_panel.add_annotation(
        text=(
            f"{ba}<br>"
            f"Mean: {mean_value:,.0f} MW<br>"
            f"Median: {median_value:,.0f} MW"
        ),
        xref="x domain",
        yref=f"y{row_number if row_number > 1 else ''} domain",
        x=0.985,
        y=0.92,
        showarrow=False,
        align="right",
        bgcolor="white",
        bordercolor="black",
        borderwidth=1,
        font=dict(size=10),
        row=row_number,
        col=1
    )

fig_daily_peak_panel.show()

#### Step 13C Summary

The daily peak demand panel view improves interpretation by separating each California balancing authority into its own chart area. This avoids the scale problem from the combined daily peak chart, where CISO dominates the y-axis and compresses the smaller authorities.

The red dashed line shows the mean daily peak demand, while the green dotted line shows the median daily peak demand. Comparing these two reference lines helps identify whether peak demand is stable or influenced by occasional spikes. When the mean is higher than the median, the authority likely has a few unusually high peak days that pull the average upward.

BANC shows the clearest spike-driven behavior. Its mean daily peak demand is noticeably above its median, which reflects several sharp peak days during the study period. CISO’s mean and median are close, suggesting a more stable daily peak pattern despite its much larger overall scale. IID and TIDC show seasonal or period-specific increases around March, while LDWP shows a more stable mid-range pattern but also includes very low values at the beginning and end of the period.

This chart supports the stress index design because it confirms that raw demand levels alone are not enough for fair comparison across authorities. Each authority has a different operating scale and different peak behavior, so later stress features should be normalized within each balancing authority.

### Step 14: Additional EDA for Grid Behavior

The previous demand visualizations showed that California balancing authorities differ strongly in scale, peak behavior, and daily demand patterns. Before creating the stress index, the analysis now examines additional operational signals that help explain grid behavior beyond demand alone.

This section reviews forecast accuracy, the relationship between demand and net generation, interchange patterns, hourly demand profiles, heatmap-based time patterns, and day-of-week demand variation. These views provide the practical foundation for designing a stress index that reflects multiple grid conditions rather than relying on raw demand alone.

In [34]:
# ============================================
# STEP 14A: COMPARE ACTUAL DEMAND AND DEMAND FORECAST
# ============================================

# Define visual order from top to bottom
ba_order = ["BANC", "CISO", "IID", "LDWP", "TIDC"]

# Sort data for cleaner time-series plotting
df_ca_sorted = df_ca.sort_values(
    ["balancing_authority", "local_time_pacific"]
)

# Create panel chart comparing actual demand and forecasted demand
fig_demand_vs_forecast = px.line(
    df_ca_sorted,
    x="local_time_pacific",
    y=["demand_mw", "demand_forecast_mw"],
    facet_row="balancing_authority",
    category_orders={"balancing_authority": ba_order},
    title="Actual Demand vs Forecasted Demand by California Balancing Authority",
    labels={
        "local_time_pacific": "Local Time (Pacific)",
        "value": "Megawatts (MW)",
        "variable": "Series"
    },
    height=1050
)

# Clean facet labels
fig_demand_vs_forecast.for_each_annotation(
    lambda a: a.update(text=a.text.split("=")[-1])
)

# Allow each balancing authority to use its own y-axis scale
fig_demand_vs_forecast.update_yaxes(matches=None)

fig_demand_vs_forecast.show()

In [35]:
# ============================================
# STEP 14A CHECK: FORECAST ERROR SUMMARY
# ============================================

# Create forecast error fields
df_ca["forecast_error_mw"] = df_ca["demand_mw"] - df_ca["demand_forecast_mw"]
df_ca["abs_forecast_error_mw"] = df_ca["forecast_error_mw"].abs()

# Summarize forecast error by balancing authority
forecast_error_summary = (
    df_ca
    .groupby("balancing_authority", as_index=False)
    .agg(
        observations=("forecast_error_mw", "count"),
        mean_error_mw=("forecast_error_mw", "mean"),
        median_error_mw=("forecast_error_mw", "median"),
        mean_abs_error_mw=("abs_forecast_error_mw", "mean"),
        max_abs_error_mw=("abs_forecast_error_mw", "max")
    )
    .round(2)
)

print("Forecast error summary by balancing authority:")
display(forecast_error_summary)

Forecast error summary by balancing authority:


#### Step 14A Summary

The actual demand and forecast comparison shows that forecast quality differs across California balancing authorities. IID and TIDC have very small forecast errors, with mean absolute errors of about **10.87 MW** and **8.83 MW**, respectively. Their actual and forecasted demand lines closely overlap for most of the observed period.

CISO shows the largest forecast gap. Its mean error is about **2,234 MW**, and its mean absolute error is about **2,351 MW**. This means actual CISO demand was generally higher than forecasted demand during the study period. CISO also has the largest maximum absolute forecast error at **10,573 MW**, which suggests that some periods had substantial planning deviation.

BANC has a small average forecast error, but its maximum absolute error reaches **7,616 MW**, likely due to isolated spike behavior rather than consistent forecast mismatch. LDWP shows moderate forecast error, with a mean absolute error of about **462 MW**, and its forecast line does not fully capture some demand shifts, especially near the beginning and end of the observed period.

This step shows that forecast error is an important operational signal. For the later stress index, forecast deviation should be considered alongside demand level, because high demand is more concerning when it is also under-forecast or poorly anticipated.

#### Step 14B: Compare Demand and Net Generation

Demand is compared with net generation to examine whether local generation is moving with electricity use. When demand is higher than net generation, the difference is often supported by interchange with neighboring systems.

This comparison helps explain grid balance. It also supports the stress index design by showing that high demand alone is not the full story. A high-demand period may be more operationally important when local generation is low or when the authority depends more heavily on interchange.

In [36]:
# ============================================
# STEP 14B: COMPARE DEMAND AND NET GENERATION
# ============================================

# Sort data for cleaner time-series plotting
df_ca_sorted = df_ca.sort_values(
    ["balancing_authority", "local_time_pacific"]
)

# Create panel chart comparing demand and net generation
fig_generation_vs_demand = px.line(
    df_ca_sorted,
    x="local_time_pacific",
    y=["demand_mw", "net_generation_mw"],
    facet_row="balancing_authority",
    category_orders={"balancing_authority": ba_order},
    title="Demand vs Net Generation by California Balancing Authority",
    labels={
        "local_time_pacific": "Local Time (Pacific)",
        "value": "Megawatts (MW)",
        "variable": "Series"
    },
    height=1050
)

# Clean facet labels
fig_generation_vs_demand.for_each_annotation(
    lambda a: a.update(text=a.text.split("=")[-1])
)

# Allow each authority to use its own y-axis scale
fig_generation_vs_demand.update_yaxes(matches=None)

fig_generation_vs_demand.show()

In [37]:
# ============================================
# STEP 14B CHECK: GENERATION-DEMAND GAP SUMMARY
# ============================================

# Positive gap means demand is greater than net generation.
# This can indicate reliance on imports or other balancing mechanisms.
df_ca["generation_gap_mw"] = df_ca["demand_mw"] - df_ca["net_generation_mw"]

generation_gap_summary = (
    df_ca
    .groupby("balancing_authority", as_index=False)
    .agg(
        observations=("generation_gap_mw", "count"),
        mean_generation_gap_mw=("generation_gap_mw", "mean"),
        median_generation_gap_mw=("generation_gap_mw", "median"),
        max_generation_gap_mw=("generation_gap_mw", "max"),
        min_generation_gap_mw=("generation_gap_mw", "min")
    )
    .round(2)
)

print("Generation-demand gap summary by balancing authority:")
display(generation_gap_summary)

Generation-demand gap summary by balancing authority:


#### Step 14B Summary

The demand and net generation comparison shows that California balancing authorities differ not only in demand scale, but also in how closely local generation tracks demand.

CISO has the largest generation-demand gap. Its average gap is about **8,816 MW**, meaning demand is consistently higher than reported net generation during the observed period. This supports the earlier interchange finding that CISO often depends on power flows from outside its balancing authority to meet demand.

LDWP and BANC also show positive average generation gaps, about **780 MW** and **447 MW**, respectively. This indicates that demand is usually above net generation for these authorities as well, although at much smaller scales than CISO.

IID is different. Its average generation-demand gap is about **-290 MW**, meaning reported net generation is often higher than demand. This suggests that IID may more often operate as a net exporter or have generation patterns that exceed local demand during many hours. TIDC shows a smaller positive gap of about **92 MW**, indicating a more modest difference between demand and net generation.

This step confirms that supply balance is an important part of California grid behavior. A later stress index should not rely only on demand level. It should also account for generation-demand gap and interchange conditions, because high demand becomes more operationally important when local generation is not keeping pace.

#### Step 14C: Review Total Interchange

Total interchange shows whether a balancing authority is exchanging power with neighboring systems. Negative values generally indicate net imports, while positive values indicate net exports.

Reviewing interchange helps explain whether demand is being supported by local generation or by power flows from outside the balancing authority. This is especially important after the demand and net generation comparison, because generation gaps are often balanced through interchange.

In [38]:
# ============================================
# STEP 14C: REVIEW TOTAL INTERCHANGE
# ============================================

# Sort data for cleaner time-series plotting
df_ca_sorted = df_ca.sort_values(
    ["balancing_authority", "local_time_pacific"]
)

# Create interchange chart by balancing authority
fig_interchange = px.line(
    df_ca_sorted,
    x="local_time_pacific",
    y="total_interchange_mw",
    color="balancing_authority",
    title="Total Interchange Over Time by California Balancing Authority",
    labels={
        "local_time_pacific": "Local Time (Pacific)",
        "total_interchange_mw": "Total Interchange (MW)",
        "balancing_authority": "Balancing Authority"
    }
)

# Add zero reference line to separate net import and net export periods
fig_interchange.add_hline(
    y=0,
    line_dash="dash",
    line_color="black"
)

fig_interchange.show()

In [39]:
# ============================================
# STEP 14C CHECK: INTERCHANGE SUMMARY
# ============================================

interchange_summary = (
    df_ca
    .groupby("balancing_authority", as_index=False)
    .agg(
        observations=("total_interchange_mw", "count"),
        mean_interchange_mw=("total_interchange_mw", "mean"),
        median_interchange_mw=("total_interchange_mw", "median"),
        min_interchange_mw=("total_interchange_mw", "min"),
        max_interchange_mw=("total_interchange_mw", "max")
    )
    .round(2)
)

print("Interchange summary by balancing authority:")
display(interchange_summary)

Interchange summary by balancing authority:


#### Step 14C Summary

The total interchange chart shows clear differences in how California balancing authorities exchange power with neighboring systems. The zero reference line separates net import periods from net export periods.

CISO has the largest negative interchange pattern, with an average total interchange of about **-5,635 MW** and a minimum value of **-11,236 MW**. This means CISO is frequently a net importer during the observed period. This finding is consistent with the earlier generation-demand gap analysis, where CISO demand was consistently higher than reported net generation.

BANC, LDWP, and TIDC also show negative average interchange values, suggesting net import behavior on average. LDWP’s average interchange is about **-780 MW**, BANC’s is about **-447 MW**, and TIDC’s is about **-90 MW**. IID is different, with a positive average interchange of about **301 MW**, suggesting that it often exports more power than it imports during the study period.

This step confirms that interchange is a useful operational signal for the stress index. High demand may be more important when it occurs alongside large negative interchange, because that combination can indicate heavier reliance on external power flows.

#### Step 14D: Review Average Demand by Hour of Day

Average demand by hour of day shows the typical daily load shape for each California balancing authority. This view helps identify the hours when electricity demand is usually highest and provides useful context for stress index design.

Because balancing authorities operate at different scales, this step uses both a combined chart and a panel chart. The combined chart helps compare statewide scale, while the panel chart makes each authority’s daily shape easier to read.

In [40]:
# ============================================
# STEP 14D: REVIEW AVERAGE DEMAND BY HOUR OF DAY
# ============================================

# Add hour-of-day feature for time-based demand profiling
df_ca["hour_of_day"] = df_ca["local_time_pacific"].dt.hour

# Calculate average demand by authority and hour of day
hourly_profile = (
    df_ca
    .groupby(["balancing_authority", "hour_of_day"], as_index=False)
    .agg(avg_demand_mw=("demand_mw", "mean"))
)

print("Hourly demand profile created.")
print("Shape:", hourly_profile.shape)

display(hourly_profile.head())

# Combined hourly profile chart
fig_hourly_profile = px.line(
    hourly_profile,
    x="hour_of_day",
    y="avg_demand_mw",
    color="balancing_authority",
    title="Average Demand by Hour of Day by California Balancing Authority",
    labels={
        "hour_of_day": "Hour of Day",
        "avg_demand_mw": "Average Demand (MW)",
        "balancing_authority": "Balancing Authority"
    }
)

fig_hourly_profile.show()

Hourly demand profile created.
Shape: (120, 3)


In [42]:
# ============================================
# STEP 14D PANEL VIEW: AVERAGE DEMAND BY HOUR OF DAY
# WITH MEAN AND MEDIAN REFERENCE LINES
# ============================================

# Create hourly profile
hourly_profile = (
    df_ca
    .groupby(["balancing_authority", "hour_of_day"], as_index=False)
    .agg(avg_demand_mw=("demand_mw", "mean"))
)

# Compute mean and median of the hourly profile for each authority
hourly_profile_stats = (
    hourly_profile
    .groupby("balancing_authority", as_index=False)
    .agg(
        mean_hourly_profile_mw=("avg_demand_mw", "mean"),
        median_hourly_profile_mw=("avg_demand_mw", "median")
    )
)

ba_order = ["BANC", "CISO", "IID", "LDWP", "TIDC"]

fig_hourly_profile_panel = px.line(
    hourly_profile,
    x="hour_of_day",
    y="avg_demand_mw",
    facet_row="balancing_authority",
    category_orders={"balancing_authority": ba_order},
    title="Average Demand by Hour of Day: Panel View with Mean and Median Reference Lines",
    labels={
        "hour_of_day": "Hour of Day",
        "avg_demand_mw": "Average Demand (MW)",
        "balancing_authority": "Balancing Authority"
    },
    height=1050
)

fig_hourly_profile_panel.for_each_annotation(
    lambda a: a.update(text=a.text.split("=")[-1])
)

fig_hourly_profile_panel.update_yaxes(matches=None)

n_panels = len(ba_order)

for ba in ba_order:
    row_number = n_panels - ba_order.index(ba)

    mean_value = hourly_profile_stats.loc[
        hourly_profile_stats["balancing_authority"] == ba,
        "mean_hourly_profile_mw"
    ].iloc[0]

    median_value = hourly_profile_stats.loc[
        hourly_profile_stats["balancing_authority"] == ba,
        "median_hourly_profile_mw"
    ].iloc[0]

    fig_hourly_profile_panel.add_hline(
        y=mean_value,
        line_dash="dash",
        line_color="red",
        line_width=1.5,
        row=row_number,
        col=1
    )

    fig_hourly_profile_panel.add_hline(
        y=median_value,
        line_dash="dot",
        line_color="green",
        line_width=1.5,
        row=row_number,
        col=1
    )

    fig_hourly_profile_panel.add_annotation(
        text=(
            f"{ba}<br>"
            f"Mean: {mean_value:,.0f} MW<br>"
            f"Median: {median_value:,.0f} MW"
        ),
        xref="x domain",
        yref=f"y{row_number if row_number > 1 else ''} domain",
        x=0.985,
        y=0.92,
        showarrow=False,
        align="right",
        bgcolor="white",
        bordercolor="black",
        borderwidth=1,
        font=dict(size=10),
        row=row_number,
        col=1
    )

fig_hourly_profile_panel.show()

#### Step 14D2: Median Hourly Demand Profile with IQR Band

The previous hourly profile chart showed average demand by hour of day. This enhanced view uses the median and interquartile range, which gives a more robust picture of typical daily demand behavior.

The median line shows the typical demand level for each hour. The shaded band shows the interquartile range, from the 25th percentile to the 75th percentile. This band captures the middle 50% of observed demand values for each hour and helps show how much demand varies during different parts of the day.

This approach is useful because it is less sensitive to unusual spikes than a simple average line. It provides a stronger foundation for stress-index design by showing not only when demand is typically high, but also when hourly demand is more variable.

In [43]:
# ============================================
# STEP 14D2: MEDIAN HOURLY DEMAND PROFILE WITH IQR BAND
# ============================================

# Create hour-of-day field if it does not already exist
df_ca["hour_of_day"] = df_ca["local_time_pacific"].dt.hour

# Define visual order from top to bottom
ba_order = ["BANC", "CISO", "IID", "LDWP", "TIDC"]

# Calculate median and interquartile range by authority and hour
hourly_iqr_profile = (
    df_ca
    .groupby(["balancing_authority", "hour_of_day"], as_index=False)
    .agg(
        q25_demand_mw=("demand_mw", lambda x: x.quantile(0.25)),
        median_demand_mw=("demand_mw", "median"),
        q75_demand_mw=("demand_mw", lambda x: x.quantile(0.75))
    )
)

print("Hourly median and IQR profile created.")
print("Shape:", hourly_iqr_profile.shape)
display(hourly_iqr_profile.head())

Hourly median and IQR profile created.
Shape: (120, 5)


In [46]:
# ============================================
# STEP 14D2: MEDIAN HOURLY DEMAND PROFILE WITH IQR BAND
# AND MEAN REFERENCE LINE
# ============================================

from plotly.subplots import make_subplots
import plotly.graph_objects as go

# Create hour-of-day field if it does not already exist
df_ca["hour_of_day"] = df_ca["local_time_pacific"].dt.hour

# Define visual order from top to bottom
ba_order = ["BANC", "CISO", "IID", "LDWP", "TIDC"]

# Calculate hourly summary statistics by authority and hour
hourly_iqr_profile = (
    df_ca
    .groupby(["balancing_authority", "hour_of_day"], as_index=False)
    .agg(
        q25_demand_mw=("demand_mw", lambda x: x.quantile(0.25)),
        median_demand_mw=("demand_mw", "median"),
        mean_demand_mw=("demand_mw", "mean"),
        q75_demand_mw=("demand_mw", lambda x: x.quantile(0.75))
    )
)

print("Hourly median, mean, and IQR profile created.")
print("Shape:", hourly_iqr_profile.shape)
display(hourly_iqr_profile.head())


# ============================================
# STEP 14D2 VISUAL: MEDIAN HOURLY PROFILE WITH IQR BAND
# AND MEAN REFERENCE LINE
# ============================================

# Create one row per balancing authority
fig_iqr = make_subplots(
    rows=len(ba_order),
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.04,
    subplot_titles=ba_order
)

for row_num, ba in enumerate(ba_order, start=1):
    ba_data = (
        hourly_iqr_profile[
            hourly_iqr_profile["balancing_authority"] == ba
        ]
        .sort_values("hour_of_day")
    )

    # Upper IQR boundary
    fig_iqr.add_trace(
        go.Scatter(
            x=ba_data["hour_of_day"],
            y=ba_data["q75_demand_mw"],
            mode="lines",
            line=dict(width=0),
            showlegend=False,
            hoverinfo="skip",
            name=f"{ba} 75th percentile"
        ),
        row=row_num,
        col=1
    )

    # Lower IQR boundary with fill to previous trace
    fig_iqr.add_trace(
        go.Scatter(
            x=ba_data["hour_of_day"],
            y=ba_data["q25_demand_mw"],
            mode="lines",
            line=dict(width=0),
            fill="tonexty",
            fillcolor="rgba(255, 165, 0, 0.25)",
            showlegend=(row_num == 1),
            name="IQR band: 25th-75th percentile",
            hoverinfo="skip"
        ),
        row=row_num,
        col=1
    )

    # Mean demand line
    fig_iqr.add_trace(
        go.Scatter(
            x=ba_data["hour_of_day"],
            y=ba_data["mean_demand_mw"],
            mode="lines",
            line=dict(width=2, color="red", dash="dash"),
            showlegend=(row_num == 1),
            name="Mean demand",
            hovertemplate=(
                "Hour: %{x}<br>"
                "Mean Demand: %{y:,.0f} MW"
                "<extra></extra>"
            )
        ),
        row=row_num,
        col=1
    )

    # Median demand line
    fig_iqr.add_trace(
        go.Scatter(
            x=ba_data["hour_of_day"],
            y=ba_data["median_demand_mw"],
            mode="lines+markers",
            line=dict(width=2.5, color="rgb(31, 78, 121)"),
            marker=dict(size=4, color="rgb(31, 78, 121)"),
            showlegend=(row_num == 1),
            name="Median demand",
            hovertemplate=(
                "Hour: %{x}<br>"
                "Median Demand: %{y:,.0f} MW"
                "<extra></extra>"
            )
        ),
        row=row_num,
        col=1
    )

    # Add panel-specific y-axis title
    fig_iqr.update_yaxes(
        title_text=f"{ba}<br>MW",
        row=row_num,
        col=1
    )

# Layout formatting
fig_iqr.update_layout(
    title="Median Hourly Demand Profile with IQR Band and Mean Line by California Balancing Authority",
    height=1100,
    hovermode="x unified",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    )
)

# Apply x-axis settings to every subplot
for row_num in range(1, len(ba_order) + 1):
    fig_iqr.update_xaxes(
        range=[-0.2, 23.2],
        tickmode="linear",
        tick0=0,
        dtick=1,
        row=row_num,
        col=1
    )

# Add x-axis title to the bottom panel only
fig_iqr.update_xaxes(
    title_text="Hour of Day",
    row=len(ba_order),
    col=1
)

fig_iqr.show()

Hourly median, mean, and IQR profile created.
Shape: (120, 6)


#### Step 14D2 Summary

The median hourly demand profile with an IQR band and mean line provides a strong view of daily demand behavior across California balancing authorities. The dark blue line shows the median hourly demand, the amber band shows the interquartile range, and the red dashed line shows the mean hourly demand.

This combination helps distinguish three things at once: the typical demand level, the middle range of variation, and the effect of higher-demand values on the average. Where the mean line sits above the median line, the hourly demand pattern may be slightly right-skewed, meaning higher values are pulling the average upward.

The chart confirms that demand is strongly time-dependent across authorities. Most authorities show lower demand in early morning hours and higher demand later in the day. The IQR band also shows that some hours are more variable than others, which is useful for identifying when demand behavior is less stable.

This view is especially useful for stress-index design because it combines typical behavior, variation, and average level in one chart. It helps separate ordinary hourly demand structure from unusually high or more variable demand periods.

#### Step 14E: Demand Heatmap by Date and Hour

The demand heatmap shows how electricity demand varies by date and hour of day. This view is useful because line charts can hide repeated daily patterns. A heatmap makes it easier to identify recurring high-demand hours, unusual days, and periods when demand pressure is concentrated.

Because the project covers statewide California balancing authorities with different scales, this step begins with a focused CISO heatmap. CISO is the largest California balancing authority, so it provides the clearest view of the main statewide demand pattern. The broader California analysis still includes BANC, IID, LDWP, and TIDC in the comparison charts and summary tables.

In [51]:
# ============================================
# STEP 14E: DEMAND HEATMAP BY DATE AND HOUR
# ============================================

# Create a CISO-only subset for a focused heatmap
df_ciso = df_ca[df_ca["balancing_authority"] == "CISO"].copy()

# Create date and hour fields for heatmap structure
df_ciso["local_date"] = df_ciso["local_time_pacific"].dt.date
df_ciso["hour_of_day"] = df_ciso["local_time_pacific"].dt.hour

# Pivot data so dates are rows and hours are columns
heatmap_data = (
    df_ciso.pivot_table(
        index="local_date",
        columns="hour_of_day",
        values="demand_mw",
        aggfunc="mean"
    )
)

print("CISO heatmap data created.")
print("Shape:", heatmap_data.shape)
display(heatmap_data.head())

fig_heatmap = px.imshow(
    heatmap_data,
    aspect="auto",
    title="CISO Demand Heatmap by Date and Hour",
    labels={
        "x": "Hour of Day",
        "y": "Date",
        "color": "Demand (MW)"
    },
    color_continuous_scale="YlOrRd"
)

fig_heatmap.update_xaxes(
    tickmode="linear",
    tick0=0,
    dtick=1
)

fig_heatmap.show()

CISO heatmap data created.
Shape: (110, 24)


#### Step 14E2: Demand Heatmaps for All California Balancing Authorities

The CISO heatmap shows the largest California authority, but the statewide project also includes BANC, IID, LDWP, and TIDC. Because these authorities operate at very different demand scales, separate heatmaps are created for each balancing authority.

Separate heatmaps allow each authority’s date-and-hour demand pattern to be reviewed within its own range. This prevents CISO’s larger scale from hiding meaningful patterns in smaller authorities.

In [52]:
# ============================================
# STEP 14E2: DEMAND HEATMAPS FOR ALL CALIFORNIA BALANCING AUTHORITIES
# ============================================

# Create helper date and hour fields
df_ca["local_date"] = df_ca["local_time_pacific"].dt.date
df_ca["hour_of_day"] = df_ca["local_time_pacific"].dt.hour

# Define visual order
ba_order = ["BANC", "CISO", "IID", "LDWP", "TIDC"]

for ba in ba_order:
    ba_heatmap_data = (
        df_ca[df_ca["balancing_authority"] == ba]
        .pivot_table(
            index="local_date",
            columns="hour_of_day",
            values="demand_mw",
            aggfunc="mean"
        )
    )

    print(f"{ba} heatmap data shape:", ba_heatmap_data.shape)

    fig_ba_heatmap = px.imshow(
        ba_heatmap_data,
        aspect="auto",
        title=f"{ba} Demand Heatmap by Date and Hour",
        labels={
            "x": "Hour of Day",
            "y": "Date",
            "color": "Demand (MW)"
        },
        color_continuous_scale="YlOrRd"
    )

    fig_ba_heatmap.update_xaxes(
        tickmode="linear",
        tick0=0,
        dtick=1
    )

    fig_ba_heatmap.show()

BANC heatmap data shape: (110, 24)


CISO heatmap data shape: (110, 24)


IID heatmap data shape: (110, 24)


LDWP heatmap data shape: (110, 24)


TIDC heatmap data shape: (110, 24)


#### Step 14E3: Normalized Demand Heatmaps by Balancing Authority

Raw demand heatmaps are useful for understanding each authority’s absolute demand pattern. However, raw megawatt values are not directly comparable across authorities because CISO is much larger than BANC, IID, LDWP, and TIDC.

To make the heatmaps comparable, demand is normalized within each balancing authority using percentile rank. A value near 1.00 means demand is high relative to that authority’s own observed range, while a value near 0.00 means demand is low relative to that authority’s own history.

This view is especially useful before building the stress index because the stress index should compare abnormal pressure within each authority, not simply reward the largest authority for having the largest raw demand.

In [53]:
# ============================================
# STEP 14E3: NORMALIZED DEMAND HEATMAPS BY BALANCING AUTHORITY
# ============================================

# Create a normalized copy for heatmap comparison
df_ca_heat = df_ca.copy()

df_ca_heat["local_date"] = df_ca_heat["local_time_pacific"].dt.date
df_ca_heat["hour_of_day"] = df_ca_heat["local_time_pacific"].dt.hour

# Percentile-rank demand within each balancing authority
df_ca_heat["demand_percentile_within_ba"] = (
    df_ca_heat
    .groupby("balancing_authority")["demand_mw"]
    .rank(pct=True)
)

for ba in ba_order:
    ba_norm_heatmap_data = (
        df_ca_heat[df_ca_heat["balancing_authority"] == ba]
        .pivot_table(
            index="local_date",
            columns="hour_of_day",
            values="demand_percentile_within_ba",
            aggfunc="mean"
        )
    )

    print(f"{ba} normalized heatmap data shape:", ba_norm_heatmap_data.shape)

    fig_norm_heatmap = px.imshow(
        ba_norm_heatmap_data,
        aspect="auto",
        title=f"{ba} Normalized Demand Heatmap by Date and Hour",
        labels={
            "x": "Hour of Day",
            "y": "Date",
            "color": "Demand Percentile"
        },
        color_continuous_scale="YlOrRd",
        zmin=0,
        zmax=1
    )

    fig_norm_heatmap.update_xaxes(
        tickmode="linear",
        tick0=0,
        dtick=1
    )

    fig_norm_heatmap.show()

BANC normalized heatmap data shape: (110, 24)


CISO normalized heatmap data shape: (110, 24)


IID normalized heatmap data shape: (110, 24)


LDWP normalized heatmap data shape: (110, 24)


TIDC normalized heatmap data shape: (110, 24)


#### Step 14E Summary

The normalized demand heatmaps show when each California balancing authority experienced high demand relative to its own observed range. This is more useful for cross-authority comparison than raw megawatt values because CISO operates at a much larger scale than the other authorities.

Across the normalized heatmaps, higher relative demand is generally concentrated later in the day, especially from the afternoon into the evening. This confirms that demand pressure is strongly tied to hour-of-day patterns. The heatmaps also show that several authorities experienced stronger relative demand during mid-to-late March, which matches the peak behavior seen in earlier line charts.

The normalized view is especially important for smaller authorities such as IID and TIDC. In raw megawatt terms, these authorities appear small compared with CISO. But in normalized terms, their own high-demand periods become visible. This supports a fairer statewide analysis because grid stress should be measured relative to each authority’s normal operating range, not only by absolute demand.

These heatmaps provide direct support for the stress index design. A useful stress index should include within-authority normalization so that high-demand conditions in smaller authorities are not hidden by CISO’s larger scale.

#### Step 14F: Demand Distribution by Day of Week

Demand can vary by day of week because business activity, industrial activity, household routines, and weekend behavior differ across the week. This step examines whether demand levels are systematically different across weekdays and weekends.

A boxplot is used because it shows the median, spread, and high-demand outliers for each day. This provides another view of demand variability before the stress index is created.

In [54]:
# ============================================
# STEP 14F: DEMAND DISTRIBUTION BY DAY OF WEEK
# ============================================

# Add day-of-week name for weekly demand pattern analysis
df_ca["day_name"] = df_ca["local_time_pacific"].dt.day_name()

# Set calendar order for readability
day_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday"
]

fig_dow = px.box(
    df_ca,
    x="day_name",
    y="demand_mw",
    color="balancing_authority",
    category_orders={"day_name": day_order},
    title="Demand Distribution by Day of Week and Balancing Authority",
    labels={
        "day_name": "Day of Week",
        "demand_mw": "Demand (MW)",
        "balancing_authority": "Balancing Authority"
    }
)

fig_dow.show()

In [55]:
# ============================================
# STEP 14F PANEL VIEW: DEMAND DISTRIBUTION BY DAY OF WEEK
# ============================================

fig_dow_panel = px.box(
    df_ca,
    x="day_name",
    y="demand_mw",
    facet_row="balancing_authority",
    category_orders={
        "day_name": day_order,
        "balancing_authority": ba_order
    },
    title="Demand Distribution by Day of Week: Panel View",
    labels={
        "day_name": "Day of Week",
        "demand_mw": "Demand (MW)",
        "balancing_authority": "Balancing Authority"
    },
    height=1050
)

fig_dow_panel.for_each_annotation(
    lambda a: a.update(text=a.text.split("=")[-1])
)

fig_dow_panel.update_yaxes(matches=None)

fig_dow_panel.show()

#### Step 14F Summary

The day-of-week panel boxplots show how demand is distributed across weekdays and weekends for each California balancing authority. The boxplots are useful because they show the median, middle 50% range, spread, and outliers without needing additional reference lines.

CISO shows the highest demand levels across all days, with weekday demand generally appearing higher than weekend demand. This pattern is consistent with business, commercial, and industrial activity contributing more strongly during the workweek. IID and TIDC show smaller demand levels but still display visible weekday and weekend variation. LDWP shows a wider spread and several low-demand outliers, which matches earlier observations about very low reported demand during parts of the study period. BANC shows several high outliers, suggesting isolated demand spikes that should be interpreted carefully.

This chart supports the stress index design by showing that demand behavior varies not only by hour of day and date, but also by day of week. A strong stress feature should therefore preserve time-based context rather than treating all hours as equal.

#### Step 14G: Weekday vs Weekend Demand Comparison

The day-of-week boxplot provides detailed distribution by each calendar day. To simplify the pattern, this step groups days into weekday and weekend categories.

This comparison helps determine whether demand behavior changes meaningfully between workweek and weekend periods. A weekday-weekend feature may be useful later if demand patterns differ enough to support dashboard filtering or stress-index interpretation.

In [56]:
# ============================================
# STEP 14G: WEEKDAY VS WEEKEND DEMAND COMPARISON
# ============================================

# Create weekday/weekend category
df_ca["day_type"] = df_ca["local_time_pacific"].dt.dayofweek.apply(
    lambda x: "Weekend" if x >= 5 else "Weekday"
)

# Summarize demand by balancing authority and day type
weekday_weekend_summary = (
    df_ca
    .groupby(["balancing_authority", "day_type"], as_index=False)
    .agg(
        observations=("demand_mw", "count"),
        mean_demand_mw=("demand_mw", "mean"),
        median_demand_mw=("demand_mw", "median"),
        q25_demand_mw=("demand_mw", lambda x: x.quantile(0.25)),
        q75_demand_mw=("demand_mw", lambda x: x.quantile(0.75)),
        max_demand_mw=("demand_mw", "max")
    )
    .round(2)
)

print("Weekday vs weekend demand summary:")
display(weekday_weekend_summary)

Weekday vs weekend demand summary:


In [57]:
# ============================================
# STEP 14G VISUAL: WEEKDAY VS WEEKEND DEMAND PANEL
# ============================================

fig_day_type_panel = px.box(
    df_ca,
    x="day_type",
    y="demand_mw",
    facet_row="balancing_authority",
    category_orders={
        "day_type": ["Weekday", "Weekend"],
        "balancing_authority": ba_order
    },
    title="Weekday vs Weekend Demand Distribution by California Balancing Authority",
    labels={
        "day_type": "Day Type",
        "demand_mw": "Demand (MW)",
        "balancing_authority": "Balancing Authority"
    },
    height=1050
)

fig_day_type_panel.for_each_annotation(
    lambda a: a.update(text=a.text.split("=")[-1])
)

fig_day_type_panel.update_yaxes(matches=None)

fig_day_type_panel.show()

#### Step 14G Summary

The weekday versus weekend comparison shows that demand is generally higher on weekdays across all five California balancing authorities. This pattern is clearest for CISO, LDWP, and TIDC, where both mean and median demand are higher during the workweek than on weekends.

CISO has the largest weekday-weekend difference. Its average weekday demand is about **24,905 MW**, compared with about **22,904 MW** on weekends. LDWP also shows a meaningful difference, with average weekday demand of about **2,664 MW** compared with about **2,281 MW** on weekends. BANC, IID, and TIDC show smaller but still visible weekday demand increases.

This result suggests that workweek activity contributes to higher electricity demand across the California balancing authorities. It also supports the idea that `day_type` may be useful as a contextual feature or dashboard filter. However, compared with the strong hour-of-day patterns seen earlier, weekday versus weekend behavior appears to be a secondary time-based signal rather than the main driver of grid stress.

### Step 14 Summary: EDA Findings Before Stress Index Design

The exploratory analysis shows that California grid behavior cannot be summarized by demand alone. Demand varies by balancing authority, hour of day, date, and day type. CISO dominates in absolute megawatt scale, but smaller authorities such as BANC, IID, LDWP, and TIDC still show meaningful high-demand periods when measured relative to their own operating ranges.

The demand and forecast comparison showed that forecast error differs across authorities. CISO had the largest average forecast gap, while IID and TIDC had much smaller forecast errors. This suggests that forecast deviation may be an important operational signal, especially for larger or more variable authorities.

The demand and net generation comparison showed that supply balance matters. CISO, LDWP, BANC, and TIDC generally had demand above net generation, while IID often showed net generation above demand. The interchange analysis supported this pattern: CISO had the largest negative interchange values, suggesting frequent net import reliance, while IID showed positive average interchange.

The time-based visuals added further context. The median hourly profile with the IQR band showed that demand is strongly tied to hour of day, and the normalized heatmaps showed that high relative demand appears at different times for different authorities. The weekday versus weekend analysis showed that weekday demand is generally higher, but hour-of-day behavior appears to be the stronger time-based pattern.

These findings support the next phase of the project: building a grid stress index. The stress index should combine several operational signals, including relative demand level, forecast error, generation-demand gap, interchange pressure, and time-based context. Because California balancing authorities operate at very different scales, these features should be normalized within each balancing authority before comparing stress conditions statewide.

## Step 15: Feature Engineering: Grid Stress Index

The exploratory analysis showed that grid stress should not be measured by demand alone. High demand is important, but operational pressure also depends on whether demand was forecasted accurately, whether demand exceeds net generation, whether the authority is relying on imports, and whether the hour falls into a high-demand time window.

Because California balancing authorities operate at very different scales, the stress index is calculated using within-authority normalized features. This prevents CISO from automatically dominating the index simply because it has the largest raw demand values. Instead, each authority is evaluated relative to its own observed operating range.

The stress index combines four normalized signals:

| Component | Meaning |
|---|---|
| Demand pressure | How high demand is relative to the authority’s own demand history |
| Forecast error pressure | How large the absolute forecast error is relative to the authority’s own forecast-error history |
| Generation gap pressure | How high the demand-minus-generation gap is relative to the authority’s own gap history |
| Import pressure | How large net import reliance is relative to the authority’s own interchange history |

The resulting index ranges from low to high relative stress. Higher values indicate periods that may deserve closer review in the dashboard.

In [58]:
# ============================================
# STEP 15: FEATURE ENGINEERING - GRID STRESS INDEX
# ============================================

# Create a working copy for feature engineering
df_stress = df_ca.copy()

# --------------------------------------------
# 1. Create base operational features
# --------------------------------------------

# Forecast error: positive means actual demand was higher than forecast
df_stress["forecast_error_mw"] = (
    df_stress["demand_mw"] - df_stress["demand_forecast_mw"]
)

# Absolute forecast error captures forecast miss size regardless of direction
df_stress["abs_forecast_error_mw"] = df_stress["forecast_error_mw"].abs()

# Generation gap: positive means demand is greater than net generation
df_stress["generation_gap_mw"] = (
    df_stress["demand_mw"] - df_stress["net_generation_mw"]
)

# Import pressure: negative interchange indicates net imports.
# Convert imports into positive values and exports into 0.
df_stress["import_pressure_mw"] = (
    -df_stress["total_interchange_mw"]
).clip(lower=0)

# Add time-context features
df_stress["hour_of_day"] = df_stress["local_time_pacific"].dt.hour
df_stress["day_name"] = df_stress["local_time_pacific"].dt.day_name()
df_stress["day_type"] = df_stress["local_time_pacific"].dt.dayofweek.apply(
    lambda x: "Weekend" if x >= 5 else "Weekday"
)

# --------------------------------------------
# 2. Normalize stress components within each balancing authority
# --------------------------------------------

def percentile_rank_within_group(series):
    """
    Converts a numeric series into percentile ranks from 0 to 1.
    Higher values receive higher percentile scores.
    """
    return series.rank(pct=True)

# Demand pressure: high demand relative to the authority's own history
df_stress["demand_pressure_score"] = (
    df_stress
    .groupby("balancing_authority")["demand_mw"]
    .transform(percentile_rank_within_group)
)

# Forecast error pressure: high absolute forecast error relative to own history
df_stress["forecast_error_score"] = (
    df_stress
    .groupby("balancing_authority")["abs_forecast_error_mw"]
    .transform(percentile_rank_within_group)
)

# Generation gap pressure: high demand-minus-generation gap relative to own history
df_stress["generation_gap_score"] = (
    df_stress
    .groupby("balancing_authority")["generation_gap_mw"]
    .transform(percentile_rank_within_group)
)

# Import pressure: high net import reliance relative to own history
df_stress["import_pressure_score"] = (
    df_stress
    .groupby("balancing_authority")["import_pressure_mw"]
    .transform(percentile_rank_within_group)
)

# --------------------------------------------
# 3. Combine normalized components into stress index
# --------------------------------------------

# Equal-weight index for transparent interpretation
df_stress["grid_stress_index"] = (
    0.25 * df_stress["demand_pressure_score"] +
    0.25 * df_stress["forecast_error_score"] +
    0.25 * df_stress["generation_gap_score"] +
    0.25 * df_stress["import_pressure_score"]
)

# Convert to 0-100 scale for dashboard readability
df_stress["grid_stress_score"] = df_stress["grid_stress_index"] * 100

# --------------------------------------------
# 4. Assign stress categories
# --------------------------------------------

df_stress["stress_category"] = pd.cut(
    df_stress["grid_stress_score"],
    bins=[0, 50, 75, 90, 100],
    labels=[
        "Normal",
        "Elevated",
        "High",
        "Very High"
    ],
    include_lowest=True
)

print("Grid stress index created.")
print("Dataset shape:", df_stress.shape)

print("\nStress score summary:")
display(
    df_stress[
        [
            "grid_stress_score",
            "demand_pressure_score",
            "forecast_error_score",
            "generation_gap_score",
            "import_pressure_score"
        ]
    ].describe().round(2)
)

print("\nStress category counts:")
display(
    df_stress["stress_category"]
    .value_counts()
    .rename_axis("stress_category")
    .reset_index(name="count")
)

Grid stress index created.
Dataset shape: (13020, 81)

Stress score summary:



Stress category counts:


### Step 15 Summary

The grid stress index was created successfully for the California balancing-authority dataset. The final feature-engineered dataset contains **13,020 rows and 81 columns**.

The stress index combines four within-authority normalized signals: demand pressure, forecast error pressure, generation-demand gap pressure, and import pressure. Each component was scaled within its own balancing authority so that smaller authorities such as IID and TIDC can be compared fairly with larger authorities such as CISO.

The resulting stress score ranges from about **2.02** to **99.89**, with an average score of about **50.02**. This centered distribution is expected because the component scores were based on percentile ranks. Most observations fall into the Normal or Elevated categories, while a smaller number of records are classified as High or Very High.

There are fewer non-null stress scores than total rows because some records are missing demand forecast values. Since forecast error is one of the stress-index components, those records cannot receive a complete stress score unless the missing forecast values are imputed or the index is recalculated using available components only.

The category counts show that the stress index is useful for prioritization: **175 records** are classified as Very High stress, and **1,036 records** are classified as High stress. These periods should be reviewed next to confirm whether the stress index is identifying meaningful grid conditions.

In [60]:
# ============================================
# STEP 15 CHECK: STRESS SUMMARY BY BALANCING AUTHORITY
# ============================================

stress_summary_by_ba = (
    df_stress
    .groupby("balancing_authority", as_index=False)
    .agg(
        observations=("grid_stress_score", "count"),
        mean_stress_score=("grid_stress_score", "mean"),
        median_stress_score=("grid_stress_score", "median"),
        max_stress_score=("grid_stress_score", "max"),
        very_high_periods=("stress_category", lambda x: (x == "Very High").sum()),
        high_periods=("stress_category", lambda x: (x == "High").sum()),
        elevated_periods=("stress_category", lambda x: (x == "Elevated").sum())
    )
    .round(2)
)

print("Stress summary by balancing authority:")
display(stress_summary_by_ba)

Stress summary by balancing authority:


In [61]:
# ============================================
# STEP 15 CHECK: REVIEW HIGHEST STRESS PERIODS
# ============================================

top_stress_periods = (
    df_stress
    .sort_values("grid_stress_score", ascending=False)
    [
        [
            "balancing_authority",
            "local_time_pacific",
            "hour_of_day",
            "day_name",
            "day_type",
            "demand_mw",
            "demand_forecast_mw",
            "forecast_error_mw",
            "net_generation_mw",
            "generation_gap_mw",
            "total_interchange_mw",
            "import_pressure_mw",
            "grid_stress_score",
            "stress_category"
        ]
    ]
    .head(20)
)

print("Top 20 highest stress periods:")
display(top_stress_periods)

Top 20 highest stress periods:


### Step 15 Validation Summary

The stress-index validation shows that the engineered score behaves as intended. The mean stress scores are close to 50 for all five balancing authorities, which is expected because the main stress components were normalized within each authority using percentile ranks. This confirms that the index avoids letting CISO dominate only because of its larger raw demand scale.

The highest stress periods are concentrated in records where multiple pressure signals occur together. For example, the top BANC periods combine unusually high demand, large forecast errors, large generation-demand gaps, and large negative interchange values. IID, TIDC, and LDWP also appear in the highest stress records when their own demand, generation gap, and import pressure are high relative to their normal operating ranges.

The category counts by balancing authority show that the stress index identifies authority-specific risk patterns. CISO has the largest raw demand but only one Very High stress period, while LDWP, BANC, TIDC, and IID show more Very High periods relative to their own operating behavior. This confirms that the index is not simply measuring grid size. It is measuring relative operational pressure.

This validation supports using the grid stress score as the main engineered feature for dashboard development and ranking higher-priority operating periods.

In [62]:
# ============================================
# STEP 15 FINAL CLEANUP: KEEP COMPLETE STRESS SCORE RECORDS
# ============================================

df_stress_complete = df_stress.dropna(subset=["grid_stress_score"]).copy()

print("Original stress dataset rows:", len(df_stress))
print("Complete stress-score rows:", len(df_stress_complete))
print("Rows removed due to missing stress score:", len(df_stress) - len(df_stress_complete))

Original stress dataset rows: 13020
Complete stress-score rows: 12735
Rows removed due to missing stress score: 285


### Step 15 Cleanup Summary

The complete stress-score dataset contains **12,735 records** out of the original **13,020 California records**. A total of **285 rows** were removed because they did not have a complete stress score.

The missing stress scores are primarily caused by missing demand forecast values. Since forecast error is one of the four stress-index components, those records cannot receive a fully comparable stress score without imputation or a modified scoring rule.

For this version of the project, the dashboard-ready stress dataset uses only complete stress-score records. This keeps the index transparent and ensures that every exported record is scored using the same four components: demand pressure, forecast error pressure, generation-gap pressure, and import pressure.

## Step 16: Visualize Grid Stress Index Results

After constructing and validating the grid stress index, the next step is to visualize the stress score and stress categories. These visuals help confirm whether the engineered feature behaves logically over time and whether high-stress periods are distributed across balancing authorities in a meaningful way.

This step uses the complete stress-score dataset, `df_stress_complete`, so every record shown in the visuals has all four stress-index components available: demand pressure, forecast error pressure, generation-gap pressure, and import pressure.

### Step 16A: Stress Score Over Time by Balancing Authority

The first stress-index visualization plots the grid stress score over time for each California balancing authority. A panel view is used because the authorities have different operating patterns, and separate panels make it easier to see stress movement within each authority.

This view helps identify whether stress periods are isolated spikes, repeated clusters, or broader sustained patterns.

In [64]:
# ============================================
# STEP 16A: STRESS SCORE OVER TIME BY BALANCING AUTHORITY
# WITH THRESHOLD LINES IN LEGEND
# ============================================

import plotly.graph_objects as go

df_stress_complete_sorted = df_stress_complete.sort_values(
    ["balancing_authority", "local_time_pacific"]
)

fig_stress_time = px.line(
    df_stress_complete_sorted,
    x="local_time_pacific",
    y="grid_stress_score",
    facet_row="balancing_authority",
    category_orders={"balancing_authority": ba_order},
    title="Grid Stress Score Over Time by California Balancing Authority",
    labels={
        "local_time_pacific": "Local Time (Pacific)",
        "grid_stress_score": "Grid Stress Score",
        "balancing_authority": "Balancing Authority"
    },
    height=1050
)

# Clean facet labels
fig_stress_time.for_each_annotation(
    lambda a: a.update(text=a.text.split("=")[-1])
)

# Keep same y-axis range for stress score because it is normalized
fig_stress_time.update_yaxes(range=[0, 100])

# Add threshold lines to each panel
fig_stress_time.add_hline(
    y=50,
    line_dash="dot",
    line_color="gray",
    line_width=1.5
)

fig_stress_time.add_hline(
    y=75,
    line_dash="dash",
    line_color="orange",
    line_width=1.5
)

fig_stress_time.add_hline(
    y=90,
    line_dash="dash",
    line_color="red",
    line_width=1.5
)

# Add dummy traces so threshold lines appear in the legend
# These traces are not used for data plotting. They only create legend entries.
fig_stress_time.add_trace(
    go.Scatter(
        x=[None],
        y=[None],
        mode="lines",
        line=dict(color="gray", dash="dot", width=1.5),
        name="Elevated threshold: 50",
        showlegend=True
    )
)

fig_stress_time.add_trace(
    go.Scatter(
        x=[None],
        y=[None],
        mode="lines",
        line=dict(color="orange", dash="dash", width=1.5),
        name="High threshold: 75",
        showlegend=True
    )
)

fig_stress_time.add_trace(
    go.Scatter(
        x=[None],
        y=[None],
        mode="lines",
        line=dict(color="red", dash="dash", width=1.5),
        name="Very High threshold: 90",
        showlegend=True
    )
)

# Improve legend placement
fig_stress_time.update_layout(
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    )
)

fig_stress_time.show()

### Step 16A Summary

The stress score time-series chart shows how relative grid stress changes over time for each California balancing authority. Because the stress score is normalized within each authority, the panels can be compared on the same 0–100 scale without allowing CISO’s larger raw demand to dominate the visual.

The reference lines mark the stress thresholds used in the index: **50** for Elevated, **75** for High, and **90** for Very High. Periods above these thresholds indicate hours where multiple stress signals occur together, including relative demand pressure, forecast error, generation-demand gap, and import pressure.

The chart shows that stress periods occur across all five California authorities, but the timing and pattern differ by authority. BANC and LDWP show several sharp stress spikes, while IID and TIDC show more visible elevated periods later in the study window. CISO has fewer Very High periods even though it has the largest raw demand, which confirms that the index is measuring relative operating pressure rather than system size alone.

This visualization supports dashboard development because it provides a time-based monitoring view of stress conditions across California balancing authorities.

### Step 16B: Review Priority Classification

The grid stress score is a continuous metric from 0 to 100. Continuous scores are useful for ranking records, but dashboards also need simple categories that help users filter and prioritize periods for review.

To support dashboard interpretation, the stress score is converted into a review-priority label. This label is separate from the analytical stress category. The stress category describes the full score range, while the review-priority label simplifies the result into action-oriented tiers.

| Review Priority | Stress Score Range | Interpretation |
|---|---:|---|
| High Review Priority | `>= HIGH_REVIEW_THRESHOLD` | Periods where multiple stress signals are unusually high and deserve immediate review |
| Medium Review Priority | `>= MEDIUM_REVIEW_THRESHOLD` and `< HIGH_REVIEW_THRESHOLD` | Periods with elevated stress conditions that should be monitored in trend context |
| Low Review Priority | `< MEDIUM_REVIEW_THRESHOLD` | Lower-priority periods within the observed dataset window |

The threshold values are defined as named constants so the classification logic can be adjusted without rewriting the function body.

In [65]:
# ============================================
# STEP 16B: REVIEW PRIORITY CLASSIFICATION
# ============================================

# Named constants for dashboard review-priority classification
MEDIUM_REVIEW_THRESHOLD = 75
HIGH_REVIEW_THRESHOLD = 90

def assign_review_priority(score):
    """
    Converts a continuous grid stress score into a dashboard-friendly
    review priority label.
    """
    if pd.isna(score):
        return "Missing Score"
    elif score >= HIGH_REVIEW_THRESHOLD:
        return "High Review Priority"
    elif score >= MEDIUM_REVIEW_THRESHOLD:
        return "Medium Review Priority"
    else:
        return "Low Review Priority"

# Apply review-priority classification to both stress datasets
df_stress["review_priority"] = df_stress["grid_stress_score"].apply(
    assign_review_priority
)

df_stress_complete["review_priority"] = df_stress_complete["grid_stress_score"].apply(
    assign_review_priority
)

print("Review priority classification created.")

print("\nReview priority counts:")
display(
    df_stress_complete["review_priority"]
    .value_counts()
    .rename_axis("review_priority")
    .reset_index(name="count")
)

print("\nReview priority counts by balancing authority:")
display(
    df_stress_complete
    .groupby(["balancing_authority", "review_priority"], as_index=False)
    .size()
    .rename(columns={"size": "count"})
)

Review priority classification created.

Review priority counts:



Review priority counts by balancing authority:


### Step 16B Summary

The review-priority classification was created successfully from the continuous grid stress score. The complete scored dataset contains **12,735 records**. Most records are classified as **Low Review Priority**, with **11,524 records** below the medium threshold of 75. There are **1,036 Medium Review Priority** records and **175 High Review Priority** records.

The by-authority summary shows that high-priority records are distributed across the California balancing authorities rather than being dominated by CISO. LDWP has the largest number of High Review Priority records with **69**, followed by BANC with **43**, TIDC with **35**, IID with **27**, and CISO with **1**.

This distribution supports the design of the grid stress score. Since the score is normalized within each balancing authority and combines demand pressure, forecast error pressure, generation-gap pressure, and import pressure, smaller authorities can appear in the high-priority review queue when they experience unusual pressure relative to their own operating range.

The review-priority label is now ready to support dashboard filtering, ranked review tables, and summary visuals.

### Step 16C: Review Priority Counts by Balancing Authority

This step visualizes the review-priority counts by balancing authority. The chart translates the stress score into a practical dashboard view, showing how many records from each authority fall into Low, Medium, and High Review Priority categories.

This makes the priority distribution easier to compare across authorities and supports dashboard-level filtering and monitoring.

In [66]:
# ============================================
# STEP 16C: REVIEW PRIORITY COUNTS BY BALANCING AUTHORITY
# ============================================

review_priority_counts = (
    df_stress_complete
    .groupby(["balancing_authority", "review_priority"], as_index=False)
    .size()
    .rename(columns={"size": "count"})
)

print("Review priority counts by balancing authority:")
display(review_priority_counts)

fig_review_priority = px.bar(
    review_priority_counts,
    x="balancing_authority",
    y="count",
    color="review_priority",
    barmode="group",
    category_orders={
        "balancing_authority": ba_order,
        "review_priority": [
            "Low Review Priority",
            "Medium Review Priority",
            "High Review Priority"
        ]
    },
    title="Review Priority Counts by California Balancing Authority",
    labels={
        "balancing_authority": "Balancing Authority",
        "count": "Record Count",
        "review_priority": "Review Priority"
    }
)

fig_review_priority.show()

Review priority counts by balancing authority:


### Step 16C Summary

The review-priority count chart shows that most scored records fall into the Low Review Priority category across all five California balancing authorities. This is expected because only scores at or above 75 are assigned to Medium or High Review Priority.

LDWP has the largest number of High Review Priority records with **69**, followed by BANC with **43**, TIDC with **35**, IID with **27**, and CISO with **1**. This pattern is important because it shows that high-priority periods are not determined by raw demand scale alone. CISO has the largest demand in megawatts, but it does not dominate the high-priority count.

The chart supports the dashboard design because it gives users a compact way to compare review-priority distribution by authority. It also confirms that the stress score is functioning as a relative review signal, identifying unusual pressure within each authority’s own operating range.

### Step 16C2: Review Priority Percentage by Balancing Authority

The count chart shows the number of records in each review-priority tier, but the authorities do not all have exactly the same number of complete scored observations. A percentage view makes the comparison fairer by showing the share of each authority’s records assigned to each priority tier.

In [67]:
# ============================================
# STEP 16C2: REVIEW PRIORITY PERCENTAGE BY BALANCING AUTHORITY
# ============================================

review_priority_pct = review_priority_counts.copy()

review_priority_pct["total_by_authority"] = (
    review_priority_pct
    .groupby("balancing_authority")["count"]
    .transform("sum")
)

review_priority_pct["percentage"] = (
    review_priority_pct["count"] / review_priority_pct["total_by_authority"] * 100
).round(2)

print("Review priority percentage by balancing authority:")
display(review_priority_pct)

fig_review_priority_pct = px.bar(
    review_priority_pct,
    x="balancing_authority",
    y="percentage",
    color="review_priority",
    barmode="group",
    category_orders={
        "balancing_authority": ba_order,
        "review_priority": [
            "Low Review Priority",
            "Medium Review Priority",
            "High Review Priority"
        ]
    },
    title="Review Priority Share by California Balancing Authority",
    labels={
        "balancing_authority": "Balancing Authority",
        "percentage": "Share of Scored Records (%)",
        "review_priority": "Review Priority"
    },
    text="percentage"
)

fig_review_priority_pct.update_traces(
    texttemplate="%{text:.2f}%",
    textposition="outside"
)

fig_review_priority_pct.update_yaxes(range=[0, 100])

fig_review_priority_pct.show()

Review priority percentage by balancing authority:


### Step 16C2 Summary

The review-priority percentage chart shows the share of each balancing authority’s scored records that fall into Low, Medium, and High Review Priority categories. This percentage view gives a fairer comparison than raw counts because the authorities do not all have the exact same number of complete scored observations.

Most records remain Low Review Priority for every authority. CISO has the highest Low Review Priority share at **94.50%**, with only **0.04%** of its scored records classified as High Review Priority. LDWP has the highest High Review Priority share at **2.91%** and the highest Medium Review Priority share at **12.47%**. BANC, IID, and TIDC fall between these patterns, with High Review Priority shares of **1.67%**, **1.03%**, and **1.34%**, respectively.

This chart strengthens the dashboard interpretation because it shows review-priority intensity as a percentage of each authority’s own scored records, not just as raw counts. It also confirms that the grid stress score is functioning as a relative signal rather than a raw demand ranking.

### Step 16D: Identify Top Periods for Review

A decision-maker often needs a short, ranked list rather than a full dataset. This step produces a compact review table of the highest-scoring periods across all California balancing authorities.

The table includes demand, forecast error, net generation, generation gap, interchange, import pressure, grid stress score, and review priority. These fields provide operational context so that the top-ranked periods can be interpreted as more than simple high-demand hours.

In [68]:
# ============================================
# STEP 16D: IDENTIFY TOP PERIODS FOR REVIEW
# ============================================

TOP_N_REVIEW_PERIODS = 20

top_review_periods = (
    df_stress_complete
    .sort_values("grid_stress_score", ascending=False)
    .loc[:, [
        "local_time_pacific",
        "balancing_authority",
        "hour_of_day",
        "day_name",
        "day_type",
        "demand_mw",
        "demand_forecast_mw",
        "forecast_error_mw",
        "net_generation_mw",
        "generation_gap_mw",
        "total_interchange_mw",
        "import_pressure_mw",
        "grid_stress_score",
        "review_priority"
    ]]
    .head(TOP_N_REVIEW_PERIODS)
    .reset_index(drop=True)
)

top_review_periods.index += 1
top_review_periods.index.name = "Rank"

print(f"Top {TOP_N_REVIEW_PERIODS} periods for review:")
display(top_review_periods)

Top 20 periods for review:


### Step 16D Summary

The top review periods table identifies the 20 highest-scoring grid stress periods in the complete California stress dataset. All listed records are classified as High Review Priority, with grid stress scores ranging from about **97.54** to **99.89**.

The table confirms that the stress score is not simply ranking raw demand. CISO, the largest authority by megawatt demand, does not dominate the top 20 list. Instead, BANC, IID, TIDC, and LDWP appear because their stress scores are calculated relative to their own demand, forecast error, generation gap, and import pressure patterns.

Several of the highest-ranked BANC periods show a clear combination of high demand, large forecast error, large generation-demand gap, and large negative interchange. This explains why they rise to the top of the review queue. IID and TIDC also appear frequently in the top 20 when their demand and import pressure are high relative to their normal operating range.

This table is useful for dashboard development because it provides a compact review queue. It reduces the full scored dataset to a short ranked list of periods that deserve closer review, while still preserving enough operational context to explain why each record was prioritized.

### Step 16E: Visualize Top Review Periods

The top review periods are visualized as a ranked bar chart. This makes the review queue easier to scan and helps show which balancing authorities appear among the highest stress scores.

The bar chart complements the detailed table by providing a quick visual summary of the highest-priority periods.

In [69]:
# ============================================
# STEP 16E: VISUALIZE TOP REVIEW PERIODS
# ============================================

top_review_periods_plot = top_review_periods.reset_index()

fig_top_review = px.bar(
    top_review_periods_plot.sort_values("grid_stress_score", ascending=False),
    x="Rank",
    y="grid_stress_score",
    color="balancing_authority",
    hover_data=[
        "local_time_pacific",
        "hour_of_day",
        "day_name",
        "day_type",
        "demand_mw",
        "demand_forecast_mw",
        "forecast_error_mw",
        "net_generation_mw",
        "generation_gap_mw",
        "total_interchange_mw",
        "import_pressure_mw",
        "review_priority"
    ],
    title=f"Top {TOP_N_REVIEW_PERIODS} Grid Stress Periods for Review",
    labels={
        "Rank": "Review Rank",
        "grid_stress_score": "Grid Stress Score",
        "balancing_authority": "Balancing Authority"
    },
    text="grid_stress_score",
    width=1200,
    height=550
)

fig_top_review.update_traces(
    texttemplate="%{text:.1f}",
    textposition="outside"
)

fig_top_review.update_yaxes(range=[0, 105])

fig_top_review.show()

### Step 16E Summary

The ranked bar chart shows the top 20 grid stress periods in the complete California stress dataset. All of the displayed records have grid stress scores above **97.5**, meaning they are very close to the top of the 0-100 stress scale.

The chart shows that BANC dominates the highest ranks, especially the first several positions. IID, TIDC, and LDWP also appear in the top 20. CISO does not appear in this top review chart, even though it has the largest raw demand in megawatts. This confirms that the grid stress score is not simply a raw demand ranking.

This visualization supports the project’s main design principle: review priority should be based on relative operational pressure within each balancing authority. By using normalized stress components, the review queue can surface high-pressure periods from smaller authorities that would be hidden in a raw megawatt ranking.

In [71]:
# ============================================
# STEP 16E2: ZOOMED VIEW OF TOP REVIEW PERIODS
# ============================================

fig_top_review_zoom = px.bar(
    top_review_periods_plot.sort_values("grid_stress_score", ascending=False),
    x="Rank",
    y="grid_stress_score",
    color="balancing_authority",
    hover_data=[
        "local_time_pacific",
        "hour_of_day",
        "day_name",
        "day_type",
        "demand_mw",
        "demand_forecast_mw",
        "forecast_error_mw",
        "net_generation_mw",
        "generation_gap_mw",
        "total_interchange_mw",
        "import_pressure_mw",
        "review_priority"
    ],
    title=f"Top {TOP_N_REVIEW_PERIODS} Grid Stress Periods for Review: Zoomed View",
    labels={
        "Rank": "Review Rank",
        "grid_stress_score": "Grid Stress Score",
        "balancing_authority": "Balancing Authority"
    },
    text="grid_stress_score",
    width=1200,
    height=550
)

fig_top_review_zoom.update_traces(
    texttemplate="%{text:.1f}",
    textposition="outside",
    cliponaxis=False
)

fig_top_review_zoom.update_yaxes(
    range=[95, 100.35]   # give extra headroom above tallest bar
)

fig_top_review_zoom.update_layout(
    margin=dict(t=100, r=40, l=60, b=60),   # more room at the top
    uniformtext_minsize=10,
    uniformtext_mode="show"
)

fig_top_review_zoom.show()

### Step 16E2 Summary

The zoomed top review chart provides a clearer view of the highest-ranked grid stress periods. All top 20 records have stress scores above **97.5**, so the zoomed y-axis helps show the small differences between records that would be harder to see on a full 0–100 scale.

BANC occupies the highest-ranked positions, followed by IID, TIDC, and LDWP. CISO does not appear in the top 20, even though it has the largest raw demand. This confirms that the grid stress score is not a raw megawatt ranking. Instead, it ranks periods based on relative pressure within each balancing authority.

This chart is useful for dashboard storytelling because it turns the stress index into a practical review queue. A user can quickly identify the highest-priority periods and then use the supporting table to inspect demand, forecast error, generation gap, and interchange conditions for each ranked hour.

## Step 17: Key Findings Summary

This section calculates the main project findings directly from the complete stress-score dataset. These values can be used in the project README, dashboard description, portfolio write-up, or executive summary.

The summary includes the total number of California records, the number of complete scored records, review-priority counts, peak demand, average grid stress score, maximum grid stress score, and the balancing authority with the highest number of high-priority review periods.

Calculating these values directly in the notebook creates a reproducible audit trail. If the dataset or stress-index logic changes later, the summary can be regenerated automatically instead of being updated manually.

In [72]:
# ============================================
# STEP 17: KEY FINDINGS SUMMARY
# ============================================

# Total records in the California-only dataset
total_california_records = len(df_ca)

# Total records with complete grid stress scores
total_scored_records = len(df_stress_complete)

# Review-priority counts
priority_counts = df_stress_complete["review_priority"].value_counts()

high_count = priority_counts.get("High Review Priority", 0)
medium_count = priority_counts.get("Medium Review Priority", 0)
low_count = priority_counts.get("Low Review Priority", 0)

# Review-priority percentages
high_pct = round((high_count / total_scored_records) * 100, 2)
medium_pct = round((medium_count / total_scored_records) * 100, 2)
low_pct = round((low_count / total_scored_records) * 100, 2)

# Peak demand record
peak_row = df_stress_complete.loc[df_stress_complete["demand_mw"].idxmax()]
peak_ba = peak_row["balancing_authority"]
peak_demand = peak_row["demand_mw"]
peak_time = peak_row["local_time_pacific"]

# Stress score summary
avg_stress_score = round(df_stress_complete["grid_stress_score"].mean(), 2)
max_stress_score = round(df_stress_complete["grid_stress_score"].max(), 2)

# Balancing authority with the most high-priority periods
high_priority_by_ba = (
    df_stress_complete[
        df_stress_complete["review_priority"] == "High Review Priority"
    ]
    .groupby("balancing_authority")
    .size()
    .sort_values(ascending=False)
)

top_high_priority_ba = high_priority_by_ba.index[0]
top_high_priority_count = high_priority_by_ba.iloc[0]

# Print key findings
print("=== Key Findings: California Grid Dataset (Jan-Apr 2026) ===")
print(f"Total California records:                  {total_california_records:,}")
print(f"Complete scored records:                   {total_scored_records:,}")
print(f"Records excluded from scoring:             {total_california_records - total_scored_records:,}")
print()
print(f"High Review Priority periods:              {high_count:,}  ({high_pct:.2f}% of scored records)")
print(f"Medium Review Priority periods:            {medium_count:,}  ({medium_pct:.2f}% of scored records)")
print(f"Low Review Priority periods:               {low_count:,}  ({low_pct:.2f}% of scored records)")
print()
print(f"Peak demand single hour:                   {peak_demand:,.0f} MW - {peak_ba}")
print(f"Peak demand timestamp:                     {peak_time}")
print()
print(f"Average grid stress score:                 {avg_stress_score}")
print(f"Maximum grid stress score:                 {max_stress_score}")
print()
print(
    "Authority with most High Review Priority periods: "
    f"{top_high_priority_ba} ({top_high_priority_count:,})"
)

=== Key Findings: California Grid Dataset (Jan-Apr 2026) ===
Total California records:                  13,020
Complete scored records:                   12,735
Records excluded from scoring:             285

High Review Priority periods:              175  (1.37% of scored records)
Medium Review Priority periods:            1,036  (8.14% of scored records)
Low Review Priority periods:               11,524  (90.49% of scored records)

Peak demand single hour:                   35,596 MW - CISO
Peak demand timestamp:                     2026-03-20 18:00:00-07:00

Average grid stress score:                 50.02
Maximum grid stress score:                 99.89

Authority with most High Review Priority periods: LDWP (69)


### Step 17A: Visualize Review Priority Composition

After classifying each scored record into Low, Medium, or High Review Priority, the next step is to show how the scored California grid records are distributed across these categories.

A donut chart is used here because the analytical question is compositional: What share of scored records fall into each review-priority tier? This chart provides a compact summary and makes the small High Review Priority share easier to emphasize visually than a stretched bar chart.

The chart keeps the project’s semantic color scheme:
- Green = Low Review Priority
- Yellow = Medium Review Priority
- Red = High Review Priority

The center annotation highlights the High Review Priority share because that group represents the smallest but most operationally important set of records.

In [85]:
import plotly.graph_objects as go

# ============================================
# STEP 17A: REVIEW PRIORITY COMPOSITION
# ============================================

# Ensure priority order is consistent
priority_order = [
    "Low Review Priority",
    "Medium Review Priority",
    "High Review Priority"
]

priority_color_map = {
    "Low Review Priority": "#2ca02c",      # green
    "Medium Review Priority": "#e6c84f",   # yellow-gold
    "High Review Priority": "#d62728"      # red
}

priority_share_stack = priority_share.copy()

priority_share_stack["review_priority"] = pd.Categorical(
    priority_share_stack["review_priority"],
    categories=priority_order,
    ordered=True
)

priority_share_stack = priority_share_stack.sort_values("review_priority")

priority_colors = [priority_color_map[p] for p in priority_share_stack["review_priority"]]

# High-priority percentage for center KPI
high_val = priority_share_stack.loc[
    priority_share_stack["review_priority"] == "High Review Priority",
    "percentage"
].iloc[0]

fig_priority_donut = go.Figure()

fig_priority_donut.add_trace(go.Pie(
    labels=priority_share_stack["review_priority"],
    values=priority_share_stack["percentage"],
    hole=0.62,
    sort=False,
    direction="clockwise",
    marker=dict(
        colors=priority_colors,
        line=dict(color="white", width=2)
    ),
    textinfo="percent",
    textposition="outside",
    hovertemplate=(
        "<b>%{label}</b><br>"
        "Share: %{value:.2f}%<extra></extra>"
    ),
    pull=[0, 0, 0.12]   # emphasize High Review Priority
))

# Center annotation
fig_priority_donut.add_annotation(
    x=0.5,
    y=0.5,
    xref="paper",
    yref="paper",
    text=f"<b>{high_val:.2f}%</b><br>High Review<br>Priority",
    showarrow=False,
    font=dict(size=16, color="#d62728"),
    align="center"
)

fig_priority_donut.update_layout(
    title=dict(
        text="Operational Risk: Review Priority Composition",
        x=0.5,
        xanchor="center"
    ),
    legend=dict(
        orientation="h",
        yanchor="top",
        y=-0.08,
        xanchor="center",
        x=0.5,
        title="Review Priority"
    ),
    width=700,
    height=560,
    margin=dict(t=90, b=110, l=60, r=60)
)

fig_priority_donut.show()

### Step 17A Summary

The donut chart shows that the California grid dataset is dominated by Low Review Priority periods, which account for 90.49% of scored records. Medium Review Priority periods account for 8.14%, while High Review Priority periods account for only 1.37%.

This is an important result. The review-priority system sharply narrows the dataset from thousands of scored hourly records to a very small subset of periods that deserve immediate operational attention. In other words, the grid stress framework is not merely descriptive; it functions as a triage tool.

The highlighted High Review Priority segment is small by design. That is desirable, because the purpose of the classification is to identify a limited number of exceptional periods rather than label a large share of records as critical.

### Step 17B: High-Priority Risk Concentration by Authority

After reviewing the overall priority composition, this step focuses only on High Review Priority periods. The goal is to identify which California balancing authorities contributed the largest number of high-priority records.

A horizontal bar chart is used because it makes the ranking easy to read. Sorting the authorities by count places the highest concentration at the top of the chart, making the main finding immediately visible.

In [88]:
# ============================================
# STEP 17B: HIGH-PRIORITY RISK CONCENTRATION BY AUTHORITY
# ============================================

# Count High Review Priority periods by balancing authority
high_priority_counts = (
    df_stress_complete[
        df_stress_complete["review_priority"] == "High Review Priority"
    ]
    .groupby("balancing_authority", as_index=False)
    .size()
    .rename(columns={"size": "high_priority_count"})
)

# Sort so the largest value appears at the top of the horizontal chart
high_priority_counts = high_priority_counts.sort_values(
    "high_priority_count",
    ascending=True
)

fig_high_priority = px.bar(
    high_priority_counts,
    y="balancing_authority",
    x="high_priority_count",
    orientation="h",
    title="High-Priority Risk Concentration by Authority",
    labels={
        "balancing_authority": "Balancing Authority",
        "high_priority_count": "High Review Priority Periods"
    },
    color="high_priority_count",
    color_continuous_scale="Reds",
    text="high_priority_count",
    width=850,
    height=450
)

fig_high_priority.update_traces(
    textposition="outside",
    cliponaxis=False,
    marker_line_color="rgb(8,48,107)",
    marker_line_width=1.5,
    opacity=0.85,
    width=0.6
)

fig_high_priority.update_xaxes(
    showgrid=True,
    gridcolor="LightGray",
    range=[0, high_priority_counts["high_priority_count"].max() * 1.18]
)

fig_high_priority.update_layout(
    plot_bgcolor="white",
    yaxis=dict(title=""),
    showlegend=False,
    coloraxis_showscale=False,
    margin=dict(l=100, r=60, t=80, b=50)
)

fig_high_priority.show(config={"responsive": False})

### Step 17B Summary

The high-priority risk concentration chart shows how High Review Priority periods are distributed across California balancing authorities. LDWP has the largest number of High Review Priority periods with 69 records, followed by BANC with 43, TIDC with 35, IID with 27, and CISO with 1.

This result reinforces the main purpose of the grid stress score. CISO has the largest raw demand in megawatts, but it does not dominate the high-priority review queue. The score identifies relative operational pressure within each authority by combining demand pressure, forecast error pressure, generation-demand gap pressure, and import pressure.

This chart is useful for executive reporting because it quickly shows where the highest-priority review burden is concentrated during the study period.

### Step 17 Summary

The key findings summary confirms that the California-only dataset contains **13,020 records**, of which **12,735 records** have complete grid stress scores. A total of **285 records** were excluded from scoring because at least one required stress-index component was missing, primarily due to missing demand forecast values.

Most scored records fall into the Low Review Priority category. There are **11,524 Low Review Priority periods**, representing **90.49%** of the scored dataset. There are **1,036 Medium Review Priority periods**, representing **8.14%**, and **175 High Review Priority periods**, representing **1.37%**. This distribution is useful because it creates a focused review queue instead of treating all hourly records as equally important.

The highest raw demand hour occurred in CISO at **35,596 MW** on **March 20, 2026 at 6:00 PM Pacific Time**. However, the authority with the most High Review Priority periods is **LDWP**, with **69 high-priority periods**. This contrast reinforces the value of the grid stress score: the project is not simply ranking the largest raw demand values, but identifying relative operational pressure within each authority.

The average grid stress score is **50.02**, and the maximum score is **99.89**. These values are consistent with the percentile-based scoring design and confirm that the final stress score can support dashboard ranking, filtering, and review-priority analysis.

## Step 18: Save Core Analysis Output Tables

This step saves the two core analysis output tables from the completed grid stress workflow.

The first file is the complete California grid stress dataset. It preserves the cleaned California records with engineered fields such as forecast error, generation gap, import pressure, grid stress score, stress category, and review priority. This file is the main processed analytical dataset.

The second file is the ranked top review periods table. It provides a compact review queue of the highest-scoring grid stress periods, including the supporting operational fields needed to explain why each period was prioritized.

Only these two files are saved in this step because they represent the main analysis outputs. The dashboard-specific data model is exported separately in the next phase, where purpose-built files are created for Tableau and Power BI.

In [89]:
# ============================================
# STEP 18: SAVE FINAL OUTPUT TABLES
# ============================================

# Save ranked review table to outputs/
top_review_periods.to_csv(
    f"{OUTPUT_DIR}/top_review_periods_california.csv",
    index=True
)

# Save complete scored California grid dataset to data/processed/
df_stress_complete.to_csv(
    f"{PROCESSED_DIR}/cleaned_california_grid_stress_data.csv",
    index=False
)

print("Saved final California grid output tables successfully.")
print(f"Saved: {OUTPUT_DIR}/top_review_periods_california.csv")
print(f"Saved: {PROCESSED_DIR}/cleaned_california_grid_stress_data.csv")

print("\nExported table shapes:")
print("Top review periods:", top_review_periods.shape)
print("Complete stress dataset:", df_stress_complete.shape)

Saved final California grid output tables successfully.
Saved: ../outputs/top_review_periods_california.csv
Saved: ../data/processed/cleaned_california_grid_stress_data.csv

Exported table shapes:
Top review periods: (20, 14)
Complete stress dataset: (12735, 82)


### Step 18 Summary

The core analysis output tables were saved successfully.

The ranked review table, `top_review_periods_california.csv`, contains **20 rows and 14 columns**. This file provides a compact review queue of the highest-scoring grid stress periods, including supporting operational context such as demand, forecast error, generation gap, interchange, grid stress score, and review priority.

The complete stress dataset, `cleaned_california_grid_stress_data.csv`, contains **12,735 rows and 82 columns**. This file preserves the California-only records with complete grid stress scores and engineered fields. It is the main processed analytical dataset produced by the notebook.

These two files represent the core analysis outputs. The next phase will create separate dashboard-ready files that are smaller, purpose-built, and easier to connect to Tableau or Power BI.

## Phase 4: Dashboard Data Model Export

The core analysis outputs saved in Step 18 preserve the completed stress-score dataset and the ranked review table. This phase creates a separate dashboard data model for Tableau and the planned Power BI implementation.

Instead of connecting the dashboard to one wide analytical file, this phase exports four purpose-built CSV files:

| File | Purpose |
|---|---|
| `california_grid_dashboard_ready.csv` | Hourly detail table for time-series charts, filters, review tables, and record-level drilldowns |
| `california_grid_monthly_summary.csv` | Month-level summary table for trend analysis by balancing authority |
| `california_grid_hourly_risk_summary.csv` | Hour-of-day summary table for daily-cycle and risk-profile visuals |
| `california_grid_kpi_summary.csv` | One-row-per-authority table for dashboard KPI cards |

These files are designed to make dashboard building cleaner, faster, and easier to explain. They can be regenerated by running the notebook from top to bottom.

The dashboard data model exports four purpose-built CSV files for Tableau and the planned Power BI implementation. The exact row and column counts are printed after export so the notebook documents the final data model directly from the generated files.

In [90]:
# ============================================
# PHASE 4: DASHBOARD DATA MODEL EXPORT
# ============================================

# Work from complete scored records only
dashboard_source = df_stress_complete.copy()

# Add reusable time fields for dashboard filtering and aggregation
dashboard_source["year_month"] = dashboard_source["local_time_pacific"].dt.strftime("%Y-%m")
dashboard_source["local_date"] = dashboard_source["local_time_pacific"].dt.date
dashboard_source["hour_of_day"] = dashboard_source["local_time_pacific"].dt.hour

# ============================================
# FILE 1: DASHBOARD-READY HOURLY DETAIL
# ============================================

DASHBOARD_COLS = [
    "balancing_authority",
    "local_time_pacific",
    "local_date",
    "year_month",
    "hour_of_day",
    "day_name",
    "day_type",
    "demand_mw",
    "demand_forecast_mw",
    "forecast_error_mw",
    "abs_forecast_error_mw",
    "net_generation_mw",
    "generation_gap_mw",
    "total_interchange_mw",
    "import_pressure_mw",
    "grid_stress_score",
    "stress_category",
    "review_priority"
]

df_dashboard = dashboard_source[DASHBOARD_COLS].copy()

dashboard_path = f"{PROCESSED_DIR}/california_grid_dashboard_ready.csv"
df_dashboard.to_csv(dashboard_path, index=False)

print(f"Saved: {dashboard_path}  |  shape: {df_dashboard.shape}")


# ============================================
# FILE 2: MONTHLY SUMMARY
# ============================================

monthly_summary = (
    dashboard_source
    .groupby(["balancing_authority", "year_month"], as_index=False)
    .agg(
        avg_demand_mw=("demand_mw", "mean"),
        max_demand_mw=("demand_mw", "max"),
        avg_grid_stress_score=("grid_stress_score", "mean"),
        max_grid_stress_score=("grid_stress_score", "max"),
        avg_forecast_error_mw=("forecast_error_mw", "mean"),
        avg_generation_gap_mw=("generation_gap_mw", "mean"),
        avg_import_pressure_mw=("import_pressure_mw", "mean"),
        total_scored_hours=("grid_stress_score", "count"),
        high_priority_hours=("review_priority", lambda x: (x == "High Review Priority").sum()),
        medium_priority_hours=("review_priority", lambda x: (x == "Medium Review Priority").sum()),
        low_priority_hours=("review_priority", lambda x: (x == "Low Review Priority").sum())
    )
)

for col in [
    "avg_demand_mw",
    "avg_grid_stress_score",
    "max_grid_stress_score",
    "avg_forecast_error_mw",
    "avg_generation_gap_mw",
    "avg_import_pressure_mw"
]:
    monthly_summary[col] = monthly_summary[col].round(2)

monthly_path = f"{PROCESSED_DIR}/california_grid_monthly_summary.csv"
monthly_summary.to_csv(monthly_path, index=False)

print(f"Saved: {monthly_path}  |  shape: {monthly_summary.shape}")


# ============================================
# FILE 3: HOURLY RISK SUMMARY
# ============================================

hourly_risk = (
    dashboard_source
    .groupby(["balancing_authority", "hour_of_day"], as_index=False)
    .agg(
        avg_demand_mw=("demand_mw", "mean"),
        max_demand_mw=("demand_mw", "max"),
        avg_grid_stress_score=("grid_stress_score", "mean"),
        max_grid_stress_score=("grid_stress_score", "max"),
        avg_forecast_error_mw=("forecast_error_mw", "mean"),
        avg_generation_gap_mw=("generation_gap_mw", "mean"),
        avg_import_pressure_mw=("import_pressure_mw", "mean"),
        total_observations=("grid_stress_score", "count"),
        high_priority_count=("review_priority", lambda x: (x == "High Review Priority").sum()),
        medium_priority_count=("review_priority", lambda x: (x == "Medium Review Priority").sum()),
        low_priority_count=("review_priority", lambda x: (x == "Low Review Priority").sum())
    )
)

for col in [
    "avg_demand_mw",
    "avg_grid_stress_score",
    "max_grid_stress_score",
    "avg_forecast_error_mw",
    "avg_generation_gap_mw",
    "avg_import_pressure_mw"
]:
    hourly_risk[col] = hourly_risk[col].round(2)

hourly_path = f"{PROCESSED_DIR}/california_grid_hourly_risk_summary.csv"
hourly_risk.to_csv(hourly_path, index=False)

print(f"Saved: {hourly_path}  |  shape: {hourly_risk.shape}")


# ============================================
# FILE 4: KPI SUMMARY
# ============================================

kpi_summary = (
    dashboard_source
    .groupby("balancing_authority", as_index=False)
    .agg(
        peak_demand_mw=("demand_mw", "max"),
        avg_demand_mw=("demand_mw", "mean"),
        avg_grid_stress_score=("grid_stress_score", "mean"),
        max_grid_stress_score=("grid_stress_score", "max"),
        avg_forecast_error_mw=("forecast_error_mw", "mean"),
        avg_generation_gap_mw=("generation_gap_mw", "mean"),
        avg_import_pressure_mw=("import_pressure_mw", "mean"),
        total_scored_hours=("grid_stress_score", "count"),
        date_range_start=("local_time_pacific", "min"),
        date_range_end=("local_time_pacific", "max"),
        high_priority_hours=("review_priority", lambda x: (x == "High Review Priority").sum()),
        medium_priority_hours=("review_priority", lambda x: (x == "Medium Review Priority").sum()),
        low_priority_hours=("review_priority", lambda x: (x == "Low Review Priority").sum())
    )
)

for col in [
    "avg_demand_mw",
    "avg_grid_stress_score",
    "max_grid_stress_score",
    "avg_forecast_error_mw",
    "avg_generation_gap_mw",
    "avg_import_pressure_mw"
]:
    kpi_summary[col] = kpi_summary[col].round(2)

kpi_summary["pct_high_priority"] = (
    kpi_summary["high_priority_hours"] / kpi_summary["total_scored_hours"] * 100
).round(2)

kpi_summary["pct_medium_priority"] = (
    kpi_summary["medium_priority_hours"] / kpi_summary["total_scored_hours"] * 100
).round(2)

kpi_summary["pct_low_priority"] = (
    kpi_summary["low_priority_hours"] / kpi_summary["total_scored_hours"] * 100
).round(2)

kpi_summary["date_range_start"] = kpi_summary["date_range_start"].astype(str)
kpi_summary["date_range_end"] = kpi_summary["date_range_end"].astype(str)

kpi_path = f"{PROCESSED_DIR}/california_grid_kpi_summary.csv"
kpi_summary.to_csv(kpi_path, index=False)

print(f"Saved: {kpi_path}  |  shape: {kpi_summary.shape}")

print()
print("All four dashboard-ready files exported successfully.")
print("Note: These files are excluded from version control by .gitignore.")

Saved: ../data/processed/california_grid_dashboard_ready.csv  |  shape: (12735, 18)
Saved: ../data/processed/california_grid_monthly_summary.csv  |  shape: (20, 13)
Saved: ../data/processed/california_grid_hourly_risk_summary.csv  |  shape: (120, 13)
Saved: ../data/processed/california_grid_kpi_summary.csv  |  shape: (5, 17)

All four dashboard-ready files exported successfully.
Note: These files are excluded from version control by .gitignore.


### Phase 4 Export Summary

All four dashboard-ready files were exported successfully. These files are purpose-built for Tableau and the planned Power BI implementation.

| File | Rows | Columns | Role |
|---|---:|---:|---|
| `california_grid_dashboard_ready.csv` | 12,735 | 18 | Hourly detail table for time-series charts, filters, review queues, and record-level drilldowns |
| `california_grid_monthly_summary.csv` | 20 | 13 | Month-level trend table by balancing authority |
| `california_grid_hourly_risk_summary.csv` | 120 | 13 | Hour-of-day demand and stress profile for daily-cycle analysis |
| `california_grid_kpi_summary.csv` | 5 | 17 | KPI card table with one row per balancing authority |

The hourly detail file contains only complete scored records, so every exported row has the required stress-index components, grid stress score, stress category, and review priority. The monthly, hourly, and KPI summary files are derived from the same complete scored dataset, which keeps the dashboard model consistent.

These exports separate the dashboard layer from the wider analytical dataset. This makes Tableau and Power BI development cleaner because each file has a specific purpose: hourly detail, monthly trends, hour-of-day risk patterns, and KPI cards.

### Note: Published Dashboard and Notebook Versions

The export summary above accurately describes what the **current notebook code** produces: a multi-signal **Grid Stress Score** combining four normalized signals (demand pressure, forecast error pressure, generation gap pressure, and import pressure). This yields 12,735 complete records and an 18-column export.

The published **Tableau dashboard** was built on an earlier version of the pipeline using a simpler **Stress Index** defined as `(demand_mw / peak_demand_mw) × 100`. That version scored all 13,020 California hourly records and exported a 9-column dashboard-ready file.

**Relationship between the two versions:**

| Version | Records | Columns | Metric | Status |
|---|---:|---:|---|---|
| Published Tableau dashboard | 13,020 | 9 | Simple Stress Index `(demand / peak) × 100` | Live on Tableau Public |
| Current notebook export | 12,735 | 18 | Multi-signal Grid Stress Score (percentile-normalized) | Analytical enhancement |

Both versions use the same California-only scope, the same data cleaning steps, and the same review-priority classification thresholds (High ≥ 90, Medium ≥ 75). The difference is the scoring methodology and the set of records included in the dashboard export (all California records vs. complete-signal-only records).

The **README describes the published dashboard metrics** (13,020 hours, 75 High Review Priority, average Stress Index 48.62, peak demand 35,596 MW). The current notebook code represents the analytical evolution of the scoring approach.

The published Tableau dashboard remains the primary portfolio deliverable.

**Version-control note:** These generated CSV files are excluded from Git by `.gitignore`. They should be regenerated locally by running the notebook from top to bottom rather than committed to the repository.

## Step 19: Analytical Workflow Summary

This notebook implements a structured and reproducible workflow for analyzing California electricity grid conditions using public EIA-930 balancing authority data. The workflow moves from raw data ingestion to cleaned California-only records, exploratory analysis, multi-signal stress scoring, review-priority classification, and dashboard-ready exports.

The main workflow steps are:

1. **Load and inspect the raw EIA-930 dataset**  
   The notebook begins by loading the source file and reviewing its structure, column names, row count, and basic data types before transformation.

2. **Standardize and clean key fields**  
   Important columns are renamed for readability. Essential records are retained, duplicates are removed, timestamps are converted to Pacific Time, and core electricity measures are converted to numeric values.

3. **Validate analytical scope**  
   An initial full-dataset view revealed that the source file included balancing authorities from across the United States. The notebook explicitly investigates this issue before filtering to California-only records.

4. **Filter to California balancing authorities**  
   The analysis is narrowed to five California balancing authorities: BANC, CISO, IID, LDWP, and TIDC. This creates the main California working dataset.

5. **Explore California grid demand and supply patterns**  
   The notebook develops combined charts, panel charts, heatmaps, distribution plots, and weekday/weekend comparisons to study demand, forecast accuracy, generation balance, interchange behavior, hour-of-day patterns, and authority-specific differences.

6. **Engineer a multi-signal grid stress score**  
   The stress score combines four within-authority normalized components: demand pressure, forecast error pressure, generation-gap pressure, and import pressure. This design allows smaller authorities to be compared fairly with CISO.

7. **Classify review priority**  
   The continuous grid stress score is converted into Low, Medium, and High Review Priority categories using named threshold constants. These categories support dashboard filtering and review queues.

8. **Summarize key findings**  
   The notebook calculates core findings directly from the data, including scored record counts, review-priority shares, peak demand, average grid stress score, and the authority with the most High Review Priority periods.

9. **Export core analysis outputs**  
   The complete scored California stress dataset and the top review periods table are saved as reproducible project outputs.

10. **Build the dashboard data model**  
    Four purpose-built CSV files are exported for Tableau and the planned Power BI implementation: hourly detail, monthly summary, hourly risk summary, and KPI summary.

Every generated output file is excluded from version control by `.gitignore`. A reviewer with access to the raw EIA-930 data can reproduce the workflow by running this notebook from top to bottom.

## Step 20: Explore the Interactive Result

This notebook documents the analytical workflow behind the California Grid Analysis project, including data preparation, California-only scope validation, exploratory analysis, grid stress feature engineering, review-priority classification, and dashboard-ready export creation.

The exported dashboard data model is designed for Tableau and the planned Power BI implementation. The dashboard layer extends the notebook results into an interactive decision-support view focused on grid reliability, operational review, and risk-informed interpretation.

The dashboard is intended to help users explore:

- grid stress score trends over time,
- review-priority composition,
- high-priority risk concentration by balancing authority,
- hour-of-day stress patterns,
- top review periods,
- and KPI summaries by California balancing authority.

**Interactive Tableau Dashboard:**  
[Explore the California Grid Stress Monitoring Dashboard on Tableau Public](https://public.tableau.com/app/profile/sileshi.hirpa1285/viz/CaliforniaGridStressMonitoringDashboard/ExecutiveOverview)

## Step 21: Limitations and Caveats

This project uses public EIA-930 balancing authority data to build a portfolio-level California grid analysis workflow. The results are useful for analytical review, dashboard development, and decision-support demonstration, but they should be interpreted within the limits of the dataset and methodology.

### Dataset Window

The analysis covers the available January-April 2026 records in the working dataset. All summary statistics, grid stress scores, review-priority labels, and dashboard exports reflect this specific dataset window only.

### Public Data Only

This project uses public EIA-930 balancing authority data. It does not represent any utility’s internal operational systems, proprietary data pipelines, enterprise risk models, or reliability procedures.

### Grid Stress Score Is a Custom Analytical Indicator

The grid stress score is a purpose-built review indicator for this project. It combines demand pressure, forecast error pressure, generation-gap pressure, and import pressure using within-authority percentile normalization. It is not a formal operational risk score and does not replicate any utility or grid operator methodology.

### Relative, Not Absolute, Risk

The score is relative to the observed behavior of each balancing authority within this dataset window. A high score means a record is high relative to that authority’s own observed conditions, not necessarily high compared with an all-time historical record or formal reliability threshold.

### Missing Forecast Records

Records missing demand forecast values cannot receive a complete four-component stress score unless forecast values are imputed or the scoring formula is modified. This notebook uses complete stress-score records for dashboard export to keep the methodology transparent and consistent.

### No Real-Time Data Connection

The dataset is static. This pipeline does not automatically refresh as new EIA data becomes available.

### No Consequence or Impact Modeling

The project identifies periods with elevated grid stress signals. It does not model the probability, cause, or consequence of reliability events.

### Dashboard Implementation Status

The notebook exports a dashboard-ready data model for Tableau and the planned Power BI implementation. The interactive dashboard link should be added after the Tableau Public dashboard is built and published.

## Step 22: Conclusion and Portfolio Summary

### What This Notebook Demonstrates

This notebook implements an end-to-end analytical workflow for California electricity grid operations data. Starting from public EIA-930 balancing authority records, the project moves from raw data ingestion to cleaned California-only records, exploratory analysis, engineered stress scoring, review-priority classification, and dashboard-ready exports.

The workflow demonstrates the ability to:

1. load and inspect a raw public energy dataset,
2. clean and standardize key fields for analysis,
3. validate analytical scope before interpreting visualizations,
4. filter the dataset to California balancing authorities,
5. build explanatory visualizations for demand, forecast error, generation balance, interchange, and time-based patterns,
6. engineer a multi-signal grid stress score,
7. classify review priority for dashboard filtering,
8. summarize key findings directly from the data,
9. and export a structured dashboard data model for Tableau and Power BI.

### Why Scope Validation Was Important

One of the most important analytical decisions in the notebook was the scope validation step. An early full-dataset visualization ran without error, but it produced a misleading view because the source file included balancing authorities from across the United States.

This reinforced an important data-analysis principle: a chart can be technically correct and still answer the wrong question. By identifying the scope issue and filtering to California balancing authorities before continuing, the notebook keeps the analysis aligned with the project goal.

### Grid Stress Score as a Relative Review Indicator

The grid stress score is designed as a relative, multi-signal review indicator. It does not rank hours by raw demand alone. Instead, it combines four normalized components:

- demand pressure,
- forecast error pressure,
- generation-demand gap pressure,
- and import pressure.

Each component is normalized within its own balancing authority. This design allows smaller authorities such as IID and TIDC to appear in the review queue when they experience unusual pressure relative to their own operating patterns. Without this normalization, CISO would dominate most rankings because it has the largest absolute demand.

### Connection to the Dashboard Layer

The Phase 4 dashboard exports create four purpose-built CSV files for Tableau and the planned Power BI implementation:

- the hourly detail file supports time-series views, filters, review queues, and record-level drilldowns,
- the monthly summary file supports month-level trend views,
- the hourly risk summary supports hour-of-day stress profiles,
- and the KPI summary file supports per-authority dashboard scorecards.

The donut chart for review-priority composition and the horizontal bar chart for high-priority concentration by authority are intended as executive-summary dashboard visuals. They provide a quick view of overall review-priority distribution and show where High Review Priority periods are concentrated.

### Future Enhancements

Future versions of this project could improve the analysis by adding:

- a full-year or multi-year dataset for stronger seasonality analysis,
- rolling baseline features instead of fixed dataset-window percentile ranks,
- forecast-error decomposition by authority, hour, and day type,
- statistical anomaly detection using rolling z-scores or rolling percentiles,
- geographic mapping by balancing authority service territory,
- and a Power BI dashboard implementation using the same exported data model.

### Final Portfolio Statement

This project demonstrates a complete data science and business analytics workflow: data preparation, quality validation, exploratory analysis, feature engineering, prioritization logic, and dashboard data modeling.

The main value of the project is not only the final grid stress score, but the disciplined workflow behind it. The notebook shows how raw public data can be transformed into a structured analytical product that supports review, communication, and dashboard-based decision-making.

This project uses public EIA-930 data and should be interpreted as a portfolio-level analytical demonstration, not as an operational reliability system.